# APS360 Handwriting Recognition: Step 5 (webcam live demo + webcam test set)

Colab can't open your laptop's camera, so the **live demo runs on your Windows laptop** (CPU), with your phone as the webcam through DroidCam. This notebook:

| Section | What happens |
|---|---|
| Setup | Drive, code (v16), tests |
| Export the CRNN | your trained `runs/crnn_*/best.pt` → a small CPU file (`crnn_model.pt`: float32 weights + model settings + charset + preprocessing settings), checked to give identical outputs |
| Check: demo = evaluation | ~200 IAM test words run through the demo's own code, compared with Step 2's saved test predictions |
| Build `live_demo` | a self-contained folder on Drive + `live_demo.zip`: model, `live_demo.py`, `requirements.txt`, README with Windows steps, the exact training preprocessing code, 3 sample words |
| Load Model B | `outputs/model_b/crnn_model_b.pt` from Step 6 (augmented CRNN), checked to have **exactly Model A's architecture**, charset and preprocessing |
| Score the webcam test set | reads `webcam_test/` (your captured words + `labels.csv`) and runs **Model A, Model B, Tesseract and the EMNIST baseline on the same words**; 4-row table + per-model predictions + figures in `outputs/report/` |

**Runtime: CPU is fine** (no GPU needed; Runtime → Change runtime type → CPU saves GPU quota). No IAM unzip in this step.
**Time (estimate, not measured on Colab):** ~1–2 min setup + tests, ~1 min export and check, under 1 min to build the zip; scoring ~150 webcam words ~1–2 min (Tesseract install ~30 s). **About 5 min.**

**Order:** run this once now (it builds `live_demo.zip`). Collect the webcam words on your laptop, upload the `webcam_test` folder to the project folder, then **Run all again**: the scoring section runs when it finds `webcam_test/`.

## Mount Google Drive
Your Drive appears at `/content/drive/MyDrive`. Allow access when Colab asks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Paths
`PROJECT_DIR` is the Drive folder that holds `archive.zip`. If it has been moved, the cell searches your Drive for it.
Heavy files go on Colab's local disk (`/content`), which is far faster than Drive. Results are saved back to Drive.

In [ ]:
import os, subprocess
PROJECT_DIR = "/content/drive/MyDrive/University (Mech Eng)/Year 3/Fall/APS360 (Introduction to Machine Learning)/Project/Project Progress Report"
if not os.path.exists(os.path.join(PROJECT_DIR, "archive.zip")):
    print("archive.zip not found at PROJECT_DIR; searching Drive (can take a minute)...")
    hits = [h for h in subprocess.run(["find", "/content/drive/MyDrive", "-name", "archive.zip", "-size", "+500M"],
                                      capture_output=True, text=True).stdout.split("\n") if h]
    assert hits, "Could not find archive.zip anywhere in your Drive"
    PROJECT_DIR = os.path.dirname(hits[0])

ARCHIVE    = os.path.join(PROJECT_DIR, "archive.zip")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")       # results saved back to Drive
REPORT_DIR = os.path.join(OUTPUT_DIR, "report")         # figures/tables for the progress report
OFFICIAL_WORDS_TXT = os.path.join(PROJECT_DIR, "words.txt")            # full official file, if you add it
RECOVERED_WORDS_TXT = os.path.join(PROJECT_DIR, "words_recovered.txt") # made by the label-recovery step
CODE_DIR   = "/content/aps360_htr"                      # project code (this session only)
RAW_DIR    = "/content/iam_raw"                         # archive.zip unpacked here
DATA_ROOT  = "/content/iam"                             # tidy IAM layout (symlinks into RAW_DIR)
os.environ["HTR_DATA_ROOT"] = DATA_ROOT                 # src/config.py reads this
os.makedirs(REPORT_DIR, exist_ok=True)
print("PROJECT_DIR:", PROJECT_DIR)

## Write the project code
The Step 5 code is shown below as `%%writefile` cells. The model, charset and preprocessing code it reuses are identical to Steps 1-4 and come from the embedded zip.

| File | What it does |
|---|---|
| `src/demo/webcam.py` | photo clean-up (lighting, ink, tight crop, IAM-like contrast), CPU export, the `Recognizer` (training transform + greedy CTC) |
| `src/demo/live_demo.py` | the laptop app: camera detection (DirectShow on Windows), guide box, overlay, smoothing, keys, captures to `webcam_test/` |
| `src/demo/page_reader.py` | PAGE mode (beta): page finding/straightening, line + word splitting, position-based tracking, transcript |
| `src/demo/bundle.py` | builds the `live_demo` folder and zip |
| `src/demo/score.py` | reads `webcam_test/`, runs CRNN Model A / Model B / Tesseract / EMNIST on the same cleaned words; Model A vs B architecture check |
| `src/report/webcam_report.py` | 4-row webcam table, clean-up ablation table, per-model prediction CSVs, prediction grids, preprocessing figure, IAM-vs-webcam figure |

### The rest of the codebase (embedded zip)
Every step notebook is generated from one repo; this one embeds code **v16** (`aps360_htr_code_v16.zip` in your project folder).
This cell unpacks the 76 files that aren't shown below (see the other notebook or the zip to read them).
The long string is just that zip, base64-encoded.

In [ ]:
import base64, io, os, shutil, sys, zipfile
# Start from a clean copy every time: delete any older code left in /content/aps360_htr,
# and forget already-imported project modules so this session can't keep using stale code.
os.chdir("/content")
shutil.rmtree("/content/aps360_htr", ignore_errors=True)
for m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[m]
os.makedirs("/content/aps360_htr", exist_ok=True)
CODE_B64 = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).extractall("/content/aps360_htr")
with open("/content/aps360_htr/CODE_VERSION", "w") as f:
    f.write("v16\n")
print("unpacked", len(zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))).namelist()), "files | code v16")

In [ ]:
import os
for d in sorted({os.path.dirname(f) for f in ['src/demo/__init__.py', 'src/demo/webcam.py', 'src/demo/live_demo.py', 'src/demo/page_reader.py', 'src/demo/bundle.py', 'src/demo/score.py', 'src/report/webcam_report.py']}):
    os.makedirs(f'/content/aps360_htr/{d}', exist_ok=True)
%cd /content/aps360_htr

### Webcam photo → IAM-like word, and the CPU model
The same file runs in the live demo and in the scoring below.

In [ ]:
%%writefile /content/aps360_htr/src/demo/__init__.py
"""Step 5: the webcam live demo and the webcam test set."""

In [ ]:
%%writefile /content/aps360_htr/src/demo/webcam.py
"""Webcam / phone photo of a handwritten word -> the kind of image the CRNN was trained on.

Used by BOTH the live demo on the laptop (live_demo.py) and the Step 5 notebook that scores
the webcam test set, so the two always preprocess identically.

IAM word images (the training data) are tight crops of dark ink on near-white paper:
background ~250, darkest ink ~60, no margin around the ink. A photo of a word on paper is
different: grey, unevenly lit paper, a lot of empty paper around the word, sometimes ruled
lines, a shadow or the edge of the page. `clean_photo` closes that gap in five steps:

  1. grayscale;
  2. flatten the lighting: estimate the paper brightness everywhere with a max-filter that is
     wider than a pen stroke (so it removes the ink), and divide by it -> paper becomes white;
  3. find the ink with Otsu's threshold, then drop specks (tiny components), thin lines that run
     across the whole box (ruled paper) and big dark blobs coming in from the edge (shadows);
  4. crop tightly to the remaining ink (IAM crops have no margin);
  5. set the paper to IAM's background level (250); if the ink is fainter (or darker) than
     anything common in IAM, stretch it so its darkest 10% falls in IAM's range (33-110), and
     blank the paper away from the ink. Pen strokes far thinner or thicker than IAM's (relative
     to the word height) are thickened/thinned by one step.

Checked on 720 real IAM test words (already clean, so the clean-up should change little):
CRNN CER with clean_photo is within ~0.5 points of the CER without it.

The result then goes through **exactly the training transform** (src/data/transforms.py:
resize to 32 px high keeping the aspect ratio, clamp the width to 16-512 px, scale to [-1, 1]),
and the CRNN's greedy CTC decoding (src/data/charset.py). An IAM image passed with
clean=False gives exactly the tensor the evaluation used.
"""
import numpy as np
import torch
from PIL import Image

from src.config import ModelConfig
from src.data.charset import Charset
from src.data.transforms import Preprocess
from src.models.crnn import CRNN

# Measured on 400 IAM word images (median): paper 250, darkest 10% of the ink 58,
# stroke width / word height 0.049 (10th-90th percentile 0.027-0.106).
IAM_PAPER, IAM_INK = 250, 58
IAM_INK_RANGE = (33, 110)     # 5th-95th percentile was 33-125
STROKE_RANGE = (0.025, 0.11)
MODEL_FORMAT = "aps360-crnn-cpu-v1"


def _cv2():
    import cv2          # imported lazily so the model code works without OpenCV
    return cv2


def to_gray(img):
    """BGR/RGB/gray uint8 array or PIL image -> 2-D uint8 gray array."""
    cv2 = _cv2()
    if isinstance(img, Image.Image):
        return np.asarray(img.convert("L"))
    a = np.asarray(img)
    if a.ndim == 3:
        a = cv2.cvtColor(a[:, :, :3], cv2.COLOR_BGR2GRAY)
    return a.astype(np.uint8)


def flatten_lighting(gray):
    """Divide by the local paper brightness so uneven light/shadows become white paper."""
    cv2 = _cv2()
    h, w = gray.shape
    k = max(15, (min(h, w) // 3) | 1)                   # much wider than a pen stroke
    bg = cv2.dilate(gray, cv2.getStructuringElement(cv2.MORPH_RECT, (k, k)))
    bg = cv2.GaussianBlur(bg, (0, 0), k / 4).astype(np.float32)
    return np.clip(gray.astype(np.float32) / np.maximum(bg, 1) * 255, 0, 255)


def ink_mask(flat, min_contrast=25):
    """Boolean ink mask after cleanup, or None if there is no clear writing in the box."""
    cv2 = _cv2()
    u8 = flat.astype(np.uint8)
    t, _ = cv2.threshold(cv2.GaussianBlur(u8, (3, 3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (u8 < t).astype(np.uint8)
    if not m.any() or (255 - np.percentile(flat[m > 0], 10)) < min_contrast:
        return None                                      # nothing darker than the paper: empty box
    h, w = m.shape
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = np.zeros(n, bool)
    if n <= 1:
        return None
    largest = 1 + int(np.argmax(st[1:, 4]))
    for i in range(1, n):
        x, y, bw, bh, area = st[i]
        ruled_line = x == 0 and x + bw == w and bh < 0.12 * h               # thin line across the whole box
        speck = area < max(4, 0.00015 * h * w) and area < 0.03 * st[largest, 4]   # noise dot (i-dots are kept)
        keep[i] = i == largest or not (ruled_line or speck)
    m = keep[lab]
    if m.sum() < 4:
        return None
    return m


def _stroke_ratio(m):
    cv2 = _cv2()
    if m.all():                       # no background pixel at all: the distance would be "infinite"
        return 1.0
    d = cv2.distanceTransform(m.astype(np.uint8), cv2.DIST_L2, 3)
    ys = np.nonzero(m.any(1))[0]
    return 2 * float(d[m].mean()) / max(1, ys[-1] - ys[0] + 1)


def clean_photo(img, adjust_stroke=True, mask_grow=3):
    """Photo crop of one word -> IAM-like uint8 gray image (white paper, dark ink, tight crop).
    Returns None when the box has no clear writing."""
    cv2 = _cv2()
    gray = to_gray(img)
    if gray.size == 0 or min(gray.shape) < 3:
        return None
    flat = flatten_lighting(gray)
    m = ink_mask(flat)
    if m is None:
        return None
    ys, xs = np.nonzero(m)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    flat, m = flat[y0:y1, x0:x1], m[y0:y1, x0:x1]
    if adjust_stroke and m.sum() > 20:
        r = _stroke_ratio(m)
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        if r < STROKE_RANGE[0]:                          # very thin pen for the word size: thicken
            m2 = cv2.dilate(m.astype(np.uint8), k).astype(bool)
            flat = np.where(m2 & ~m, np.minimum(flat, np.percentile(flat[m], 50)), flat); m = m2
        elif r > STROKE_RANGE[1]:                        # marker-thick strokes: thin a little
            m2 = cv2.erode(m.astype(np.uint8), k).astype(bool)
            if m2.sum() > 20:
                flat = np.where(m & ~m2, np.maximum(flat, np.percentile(flat[m], 75)), flat); m = m2
    # Contrast: only pull the ink into IAM's range (darkest 10% between 33 and 110); ink that is
    # already IAM-like is left alone (forcing every word to one darkness level hurt on IAM itself).
    dark = float(np.percentile(flat[m], 10))
    target = min(max(dark, IAM_INK_RANGE[0]), IAM_INK_RANGE[1])
    out = IAM_PAPER - (255 - flat) * (IAM_PAPER - target) / max(1.0, 255 - dark)
    if mask_grow:                                        # paper away from the ink -> plain paper (no noise/texture)
        near = cv2.dilate(m.astype(np.uint8), np.ones((mask_grow, mask_grow), np.uint8)).astype(bool)
        out = np.where(near, out, IAM_PAPER)
    return np.clip(out, 0, 255).astype(np.uint8)


def guide_box(shape, wfrac=0.7, hfrac=0.25):
    """(x0, y0, x1, y1) of the centred box the word should be written in (frame shape h, w)."""
    h, w = shape[:2]
    bw, bh = int(w * wfrac), int(h * hfrac)
    x0, y0 = (w - bw) // 2, (h - bh) // 2
    return x0, y0, x0 + bw, y0 + bh


def crop(frame, box):
    x0, y0, x1, y1 = box
    return frame[y0:y1, x0:x1].copy()


# ------------------------------------------------------------------ the model on the CPU

def export_model(checkpoint, out_path, extra=None):
    """Training checkpoint (best.pt, ~60 MB with the optimizer) -> small CPU file with only
    float32 weights + the model settings + the charset + the preprocessing settings."""
    ck = torch.load(checkpoint, map_location="cpu", weights_only=False)
    from src.config import DataConfig
    d = DataConfig()
    out = {"format": MODEL_FORMAT,
           "model": {k: v.float().cpu() if v.is_floating_point() else v.cpu() for k, v in ck["model"].items()},
           "model_cfg": {k: (list(v) if isinstance(v, tuple) else v) for k, v in dict(ck["model_cfg"]).items()},
           "charset": ck["charset"],
           "preprocess": {"height": d.img_height, "min_width": d.min_width, "max_width": d.max_width},
           "info": {"epoch": int(ck.get("epoch", -1)), "best_val_cer": float(ck.get("best_cer", float("nan"))),
                    **(extra or {})}}
    torch.save(out, out_path)
    return out


class Recognizer:
    """The CRNN on the CPU. Loads the exported file (or a training best.pt)."""

    def __init__(self, model_path, device="cpu", threads=None):
        if threads:
            torch.set_num_threads(threads)
        try:
            ck = torch.load(model_path, map_location=device, weights_only=True)
        except Exception:                                # a training checkpoint (has optimizer state etc.)
            ck = torch.load(model_path, map_location=device, weights_only=False)
        mc = dict(ck["model_cfg"]); mc["cnn_channels"] = tuple(mc["cnn_channels"])
        self.model_cfg = dict(mc)                          # kept so two models can be checked for the same architecture
        self.charset = Charset(ck["charset"])
        self.model = CRNN(self.charset.num_classes, ModelConfig(**mc)).to(device).eval()
        self.model.load_state_dict(ck["model"])
        self.pre = Preprocess(**ck.get("preprocess", {}))   # the training transform (no augmentation)
        self.preprocess_cfg = {"height": self.pre.height, "min_width": self.pre.min_width,
                               "max_width": self.pre.max_width}          # effective values (defaults filled in)
        self.device = torch.device(device)
        self.info = ck.get("info", {"epoch": ck.get("epoch")})

    def tensor(self, gray_u8):
        """uint8 gray word -> (1, 32, W) tensor in [-1, 1], exactly as in training/evaluation."""
        return self.pre(Image.fromarray(gray_u8))

    @torch.no_grad()
    def predict_tensor(self, x):
        lp = self.model(x.unsqueeze(0).to(self.device))          # (T, 1, C)
        return self.charset.ctc_greedy_decode(lp[:, 0].argmax(1).tolist())

    def read(self, img, clean=True):
        """-> (text, model input as uint8 32-px image, cleaned word or None).
        clean=False: `img` is already an IAM-style word image (skips the photo clean-up)."""
        g = clean_photo(img) if clean else to_gray(img)
        if g is None:
            return "", None, None
        x = self.tensor(g)
        return self.predict_tensor(x), tensor_to_u8(x), g


def tensor_to_u8(x):
    """(1, H, W) tensor in [-1, 1] -> uint8 image (to show what the model sees)."""
    return ((x[0].numpy() + 1) * 127.5).clip(0, 255).astype(np.uint8)

### The live demo (runs on your laptop)
WORD mode (guide box, `c` to capture) and PAGE mode (beta).

In [ ]:
%%writefile /content/aps360_htr/src/demo/live_demo.py
"""Live handwriting demo: point a webcam (or a phone via DroidCam) at a word written on paper.

    python live_demo.py                 # finds a camera (tries 0-5)
    python live_demo.py --camera 1      # pick one (DroidCam is often 0 or 1)
    python live_demo.py --list          # list the cameras (name, live or placeholder), then quit
    python live_demo.py --device-name droidcam   # prefer the camera whose name contains this text
    python live_demo.py --droidcam 10.0.0.74     # read the phone app DIRECTLY over Wi-Fi (no Client)

Two modes, switched with m (the current mode is shown at the top left):

WORD mode (default, the reliable one): one word in the green guide box.
    q / Esc      quit
    space        freeze / unfreeze the picture (the frozen word is read once)
    c            capture: freezes, then TYPE THE TRUE WORD and press Enter to save it to
                 webcam_test/ (Esc cancels). Used to build the ~150-word webcam test set.
    a / d        guide box narrower / wider       w / s   guide box taller / shorter
    n            switch to the next camera that works
    m            switch to PAGE mode

PAGE mode (BETA, less accurate): a running transcript of a whole page (written live, or an
old notebook page). About twice a second the page is found, straightened, cut into lines and
words, and every word is read; a word is added to the transcript (side panel) only once its box
has been stable for a few readings (green = in the transcript, yellow = not yet).
    r            reset the transcript         s   save the transcript to transcripts/*.txt
    space        freeze / unfreeze            c   page capture: save EVERY word crop of the
                 (frozen) page into webcam_test/ with the model's guess as the label and
                 needs_check=1. Fix those labels in labels.csv and set needs_check to 0:
                 rows still marked 1 are NOT scored (a guess is not a true label).
    m            back to WORD mode            q   quit

The word inside the green box is cleaned up exactly like in the Step 5 notebook
(src/demo/webcam.py: flatten the lighting, find the ink, tight crop, IAM-like contrast), then
resized and normalized with the SAME transform as training (src/data/transforms.py), and read by
the CRNN on the CPU every few frames. The small window shows exactly what the model sees.
Testing without a camera:  python live_demo.py --image frame.png --headless
"""
import argparse
import collections
import csv
import os
import sys
import time
import traceback

# OpenCV prints pages of warnings for every camera index that doesn't exist; this demo prints its
# own clear messages instead (set OPENCV_LOG_LEVEL=WARNING to see OpenCV's again).
os.environ.setdefault("OPENCV_LOG_LEVEL", "FATAL")
HERE = os.path.dirname(os.path.abspath(__file__))
for _root in (HERE, os.path.abspath(os.path.join(HERE, "..", ".."))):   # live_demo folder, or the repo
    if os.path.isdir(os.path.join(_root, "src", "demo")):
        sys.path.insert(0, _root)
        break

import numpy as np  # noqa: E402

KEY_ESC, KEY_ENTER, KEY_ENTER2, KEY_BACK, KEY_BACK2, KEY_SPACE = 27, 13, 10, 8, 127, 32
LABELS_CSV = "labels.csv"
FIELDS = ["file", "label", "prediction", "camera", "saved_at", "frame_w", "frame_h", "needs_check"]
TARGET_WORDS = 150


# ------------------------------------------------------------------ small pure-Python pieces (tested)

class Smoother:
    """Most common prediction over the last k readings (ties -> the most recent), so the
    overlay doesn't flicker between two spellings."""

    def __init__(self, k=5):
        self.hist = collections.deque(maxlen=k)

    def add(self, text):
        self.hist.append(text)
        return self.value()

    def value(self):
        if not self.hist:
            return ""
        counts = collections.Counter(self.hist)
        best = max(counts.values())
        for t in reversed(self.hist):
            if counts[t] == best:
                return t

    def clear(self):
        self.hist.clear()


class Keys:
    """What each key does. view = 'word' or 'page'; mode = 'live', 'frozen' or 'label' (typing the
    true word, word view only). handle(key) returns an action name; typed text is in self.text."""

    WORD_KEYS = {"a": "narrower", "d": "wider", "w": "taller", "s": "shorter", "n": "next_camera", "c": "label"}
    PAGE_KEYS = {"r": "reset", "s": "save_text", "c": "page_capture", "n": "next_camera"}

    def __init__(self, view="word"):
        self.view, self.mode, self.text = view, "live", ""

    def handle(self, key):
        if key < 0 or key == 255:
            return None
        if self.mode == "label":
            if key == KEY_ESC:
                self.mode, self.text = "frozen", ""
                return "cancel"
            if key in (KEY_ENTER, KEY_ENTER2):
                if not self.text.strip():
                    return "empty"
                return "save"
            if key in (KEY_BACK, KEY_BACK2):
                self.text = self.text[:-1]
                return "edit"
            if 33 <= key <= 126:                 # printable ASCII (IAM words have no spaces)
                self.text += chr(key)
                return "edit"
            return None
        ch = chr(key).lower() if key < 256 else ""
        if key == KEY_ESC or ch == "q":
            return "quit"
        if key == KEY_SPACE:
            self.mode = "frozen" if self.mode == "live" else "live"
            return "freeze" if self.mode == "frozen" else "unfreeze"
        if ch == "m":
            self.view, self.mode = ("page" if self.view == "word" else "word"), "live"
            return "mode"
        act = (self.WORD_KEYS if self.view == "word" else self.PAGE_KEYS).get(ch)
        if act == "label":
            self.mode, self.text = "label", ""
        elif act == "page_capture":
            self.mode = "frozen"
        return act

    def saved(self):
        self.mode, self.text = "live", ""


def count_saved(out_dir):
    p = os.path.join(out_dir, LABELS_CSV)
    if not os.path.exists(p):
        return 0
    with open(p, newline="", encoding="utf-8") as f:
        return sum(1 for _ in csv.DictReader(f))


def _ensure_header(path):
    """An older labels.csv without the newest columns is rewritten with the current header."""
    if not os.path.exists(path):
        return
    with open(path, newline="", encoding="utf-8") as f:
        r = csv.DictReader(f)
        if r.fieldnames == FIELDS:
            return
        rows = list(r)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        w.writeheader()
        for row in rows:
            w.writerow({**{k: "" for k in FIELDS}, **row, "needs_check": row.get("needs_check") or "0"})


def save_capture(out_dir, crop_bgr, label, prediction="", camera="", frame_shape=(0, 0), needs_check=False):
    """Save the raw guide-box crop as images/NNNN.png and append a row to labels.csv.
    The raw crop is kept (not the cleaned one) so the notebook can redo the clean-up.
    needs_check=True marks a label that is only the model's guess (page capture): not scored until fixed.
    Returns (file name relative to out_dir, number of words saved so far)."""
    import cv2
    img_dir = os.path.join(out_dir, "images")
    os.makedirs(img_dir, exist_ok=True)
    n = 1
    while os.path.exists(os.path.join(img_dir, f"{n:04d}.png")):
        n += 1
    rel = f"images/{n:04d}.png"
    if not cv2.imwrite(os.path.join(out_dir, rel), crop_bgr):
        raise IOError(f"could not write {rel}")
    path = os.path.join(out_dir, LABELS_CSV)
    _ensure_header(path)
    new = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow({"file": rel, "label": label.strip(), "prediction": prediction, "camera": camera,
                    "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "frame_w": frame_shape[1], "frame_h": frame_shape[0], "needs_check": int(bool(needs_check))})
    return rel, count_saved(out_dir)


# ------------------------------------------------------------------ cameras and fake sources

BACKEND = "auto"          # set from --backend


def backends(choice=None):
    """OpenCV capture APIs to try, in order. auto = DirectShow, then Media Foundation, then any
    (on Windows); just 'any' elsewhere."""
    import cv2
    choice = (choice or BACKEND).lower()
    named = {"dshow": cv2.CAP_DSHOW, "msmf": cv2.CAP_MSMF, "any": cv2.CAP_ANY}
    if choice in named:
        return [named[choice]]
    return [cv2.CAP_DSHOW, cv2.CAP_MSMF, cv2.CAP_ANY] if sys.platform.startswith("win") else [cv2.CAP_ANY]


def backend():
    return backends()[0]


def open_camera(index, width=1280, height=720):
    """An opened camera that really gives frames, or None. Tries the --backend APIs in order."""
    import cv2
    for api in backends():
        cap = cv2.VideoCapture(index, api)
        if not cap.isOpened():
            cap.release()
            continue
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, width)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, height)
        for _ in range(10):                  # some cameras (DroidCam) need a few frames to start
            ok, frame = cap.read()
            if ok and frame is not None and frame.size:
                return cap
            time.sleep(0.05)
        cap.release()
    return None


MAX_CAMERAS = 6        # indices 0-5 are probed (DroidCam is often 1 or 2 when a laptop webcam exists)
NO_OTHER_CAMERA = "no other camera gives a picture"
PLACEHOLDER_HINT = ("This looks like the DroidCam placeholder: open the DroidCam Client, press Start,",
                    "make sure it shows your phone video")


DEFAULT_PREFER = "droidcam"     # DroidCam's Windows camera is called e.g. "DroidCam Video" / "DroidCam Source 3"


def camera_names():
    """(names, indexed). indexed=True: names[i] is the name of camera index i (the DirectShow order
    OpenCV uses on Windows, read with the optional 'pygrabber' package; /sys on Linux).
    indexed=False: just the camera names Windows knows (PowerShell), without index numbers.
    ([], False) if nothing could be read. Never raises."""
    try:
        if sys.platform.startswith("win"):
            try:
                from pygrabber.dshow_graph import FilterGraph
                return list(FilterGraph().get_input_devices()), True
            except Exception:
                pass
            import subprocess
            ps = ("Get-CimInstance Win32_PnPEntity | Where-Object { $_.PNPClass -eq 'Camera' -or "
                  "$_.PNPClass -eq 'Image' } | ForEach-Object { $_.Name }")
            r = subprocess.run(["powershell", "-NoProfile", "-Command", ps], capture_output=True, text=True,
                               timeout=10, creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0))
            return [x.strip() for x in r.stdout.splitlines() if x.strip()], False
        names, root = [], "/sys/class/video4linux"
        for i in range(MAX_CAMERAS):
            p = os.path.join(root, f"video{i}", "name")
            names.append(open(p).read().strip() if os.path.exists(p) else "")
        return (names, True) if any(names) else ([], False)
    except Exception:
        return [], False


def name_matches(name, wanted):
    return bool(name) and bool(wanted) and wanted.lower() in name.lower()


def preferred_order(indices, names, device_name=None):
    """Camera indices to try, best first: a name containing --device-name, then one containing
    'DroidCam', then the rest in number order. Case-insensitive; names = {index: name}."""
    idx = list(indices)

    def rank(i):
        n = names.get(i, "")
        return (0 if name_matches(n, device_name) else 1 if name_matches(n, DEFAULT_PREFER) else 2, i)
    return sorted(idx, key=rank)


def indexed_names(names, indexed):
    return {i: n for i, n in enumerate(names) if n} if indexed else {}


def probe_cameras(indices=range(MAX_CAMERAS)):
    """[(index, width, height)] for every camera index that gives frames. Each camera is released
    before the next is tried; call this only while no camera is open (DSHOW can't open a busy one)."""
    found = []
    for i in indices:
        cap = _open_or_none(i)
        if cap is not None:
            ok, frame = cap.read()
            if ok and frame is not None:
                found.append((i, frame.shape[1], frame.shape[0]))
            cap.release()
    return found


class StillDetector:
    """Spots the DroidCam placeholder. A real camera never gives the same picture twice (sensor
    noise), while DroidCam's 'not connected' output is a fixed image (its logo screen, or a flat
    light-blue / orange / green screen). looks_placeholder is True when the frames of the last
    ~1 s are all (nearly) identical, or a frame is one saturated flat colour (white paper is
    not saturated, so a blank sheet doesn't trigger it)."""

    def __init__(self, seconds=1.0, min_frames=5, thr=0.1, flat_frac=0.9):
        self.seconds, self.min_frames, self.thr, self.flat_frac = seconds, min_frames, thr, flat_frac
        self.reset()

    def reset(self):
        self.hist = collections.deque()
        self.static = self.flat = False

    @staticmethod
    def flat_colour(frame, frac=0.9):
        """True if >= frac of the pixels are one saturated colour (DroidCam's coloured screens)."""
        import cv2
        if frame.ndim != 3:
            return False
        hsv = cv2.cvtColor(np.ascontiguousarray(frame[::4, ::4]), cv2.COLOR_BGR2HSV).reshape(-1, 3).astype(np.int16)
        med = np.median(hsv, axis=0)
        if med[1] < 60:                                    # grey / white / black: not a coloured screen
            return False
        dh = np.abs(hsv[:, 0] - med[0]); dh = np.minimum(dh, 180 - dh)
        close = (dh <= 8) & (np.abs(hsv[:, 1] - med[1]) <= 40) & (np.abs(hsv[:, 2] - med[2]) <= 40)
        return bool(close.mean() >= frac)

    def add(self, frame, t=None):
        t = time.time() if t is None else t
        g = frame if frame.ndim == 2 else frame.mean(axis=2)
        small = np.asarray(g[::4, ::4], np.float32)           # subsample, no averaging: keeps the noise
        if self.hist and self.hist[-1][1].shape != small.shape:
            self.hist.clear()
        self.hist.append((t, small))
        while len(self.hist) > 2 and t - self.hist[1][0] >= self.seconds:
            self.hist.popleft()                                # keep just over `seconds` of frames
        span = t - self.hist[0][0]
        self.static = (len(self.hist) >= self.min_frames and span >= self.seconds and
                       max(float(np.abs(f - small).mean()) for _, f in self.hist) < self.thr)
        self.flat = self.flat_colour(frame, self.flat_frac)
        return self.looks_placeholder

    @property
    def looks_placeholder(self):
        return self.static or self.flat


def describe_cameras(indices=range(MAX_CAMERAS), seconds=1.2):
    """For --list: [(index, width, height, verdict)], verdict = live picture or placeholder-like."""
    out = []
    for i in indices:
        cap = _open_or_none(i)
        if cap is None:
            continue
        det, shape, t0 = StillDetector(), None, time.time()
        try:
            while time.time() - t0 < seconds:
                ok, frame = cap.read()
                if ok and frame is not None:
                    shape = frame.shape
                    det.add(frame)
                time.sleep(0.03)
        finally:
            cap.release()
        if shape is None:
            continue
        verdict = ("flat colour: looks like the DroidCam placeholder" if det.flat else
                   "static picture: looks like the DroidCam placeholder (Client not started?)" if det.static else
                   "live picture")
        out.append((i, shape[1], shape[0], verdict))
    return out


class BlankSource:
    """Stand-in when no camera opens at all, so the program keeps running (press n to retry)."""

    def __init__(self, text="no camera gives a picture: press n to retry, q to quit", shape=(480, 640)):
        self.img = np.full(shape + (3,), 40, np.uint8)
        put_text(self.img, text, (10, shape[0] // 2), 0.55)

    def read(self):
        return True, self.img.copy()

    def release(self):
        pass


def _open_or_none(i):
    try:
        return open_camera(i)
    except Exception as e:                       # a broken driver must not kill the demo
        print(f"  camera {i}: error while opening ({e})")
        return None


def switch_camera(src, current, indices=range(MAX_CAMERAS), retries=3, pause=0.2, names=None):
    """Release `src`, then try the indices after `current` in turn (wrapping round). Returns
    (new_source, new_index, message). Never returns None as the source: if no other index gives
    frames the previous camera is re-opened (message NO_OTHER_CAMERA); if even that fails, a
    BlankSource that says 'press n to retry'."""
    if src is not None:
        try:
            src.release()
        except Exception:
            pass
        time.sleep(pause)                        # DSHOW needs a moment before the device is free again
    idx = list(indices)
    start = idx.index(current) + 1 if current in idx else 0
    order = [i for i in idx[start:] + idx[:start] if i != current]
    for i in order:
        print(f"  trying camera {i} ...")
        cap = _open_or_none(i)
        if cap is not None:
            label = f' "{names[i]}"' if names and i in names else ""
            print(f"switched to camera {i}{label}")
            return cap, i, f"switched to camera {i}{label}"
    if current is not None:
        for _ in range(retries):
            cap = _open_or_none(current)
            if cap is not None:
                print(f"{NO_OTHER_CAMERA}: staying on camera {current}")
                return cap, current, f"{NO_OTHER_CAMERA} (still camera {current})"
            time.sleep(pause)
    print("no camera gives a picture at all: check the DroidCam Client, then press n to retry")
    return BlankSource(), current, "no camera gives a picture: press n to retry"


# ------------------------------------------------------------------ DroidCam over Wi-Fi (MJPEG over HTTP)

DROIDCAM_PORT = 4747
# The phone's own default size first (a requested size can make some phones send a scrambled
# picture). Addresses that answer 404 are dropped after one try, so the extra ones cost nothing.
DROIDCAM_PATHS = ("/video", "/video?1280x720", "/mjpegfeed", "/video?640x480")
DROIDCAM_SHOTS = ("/cam/1/frame.jpg", "/shot.jpg")       # one JPEG per request (slower fallback)
CHECK_FRAMES = 6            # frames looked at on connect to decide if a URL's picture is scrambled
BUSY_HELP = ("The phone's DroidCam app says it is BUSY (another program is using the video). Fix:\n"
             "  1) Quit the DroidCam Client on the laptop completely: right-click its icon in the system tray\n"
             "     (bottom right, near the clock, maybe under the ^ arrow) -> Exit; or Task Manager -> DroidCam -> End task.\n"
             "  2) Close every browser tab that shows the phone video.\n"
             "  3) On the phone, close the DroidCam app and open it again.\n"
             "  Then run this again.")
NET_HELP = ("Can't reach the phone at {url}. Check:\n"
            "  - the phone and the laptop are on the SAME Wi-Fi (not a guest network, no VPN),\n"
            "  - the DroidCam app is open on the phone with the screen on,\n"
            "  - the IP is exactly the 'WiFi IP' shown in the app (it can change after a reconnect).")
NOJPEG_HELP = ("The phone answered, but not with JPEG pictures this program can read.\n"
               "  - In the DroidCam phone app: Settings -> set the video format/quality to the defaults\n"
               "    (Quality 'Normal', Advanced Options OFF), close the app and open it again.\n"
               "  - Please copy the 'DroidCam check' lines above when asking for help.")
GARBLED_HELP = ("The phone's picture looks SCRAMBLED (copies side by side, green lines). It is shown anyway.\n"
                "  - If the demo window actually looks FINE, ignore this (or run with --no-garble-check).\n"
                "  - Please send the saved picture {sample} when asking for help.\n"
                "  - If it really is scrambled: close and reopen the DroidCam app, switch to the back camera,\n"
                "    set its video Quality to 'Normal', or restart the phone.")
GARBLED_NOTE = "phone picture looks scrambled? see the terminal"


def is_stream_url(s):
    return isinstance(s, str) and s.lower().startswith(("http://", "https://"))


def droidcam_urls(target, paths=DROIDCAM_PATHS):
    """'10.0.0.74', '10.0.0.74:4747' or a full URL -> URLs to try, best first. A URL with its own
    path is used as given; without a path the DroidCam paths (phone default size first) are tried."""
    t = target.strip()
    if not is_stream_url(t):
        t = "http://" + t
    from urllib.parse import urlsplit
    u = urlsplit(t)
    host = u.hostname or ""
    port = u.port or DROIDCAM_PORT
    base = f"{u.scheme}://{host}:{port}"
    if u.path not in ("", "/") or u.query:
        return [base + u.path + (("?" + u.query) if u.query else "")]
    return [base + p for p in paths]


def droidcam_shot_urls(target):
    """Single-picture URLs on the same phone (used when the video stream can't be read)."""
    base = droidcam_urls(target)[0].split("/", 3)
    return ["/".join(base[:3]) + p for p in DROIDCAM_SHOTS]


def decode_jpeg(data):
    """Bytes -> BGR image, ONLY if they are one complete JPEG (starts FFD8, ends FFD9) that OpenCV
    decodes. Anything else (raw camera buffers, cut-off or broken JPEGs) gives None: raw bytes are
    never reshaped into a picture."""
    import cv2
    data = bytes(data).rstrip(b"\r\n \t")
    if len(data) < 200 or not data.startswith(b"\xff\xd8") or not data.endswith(b"\xff\xd9"):
        return None
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None or img.ndim != 3 or min(img.shape[:2]) < 32:
        return None
    return img


def garble_scores(img):
    """(typical change between neighbouring rows, ... between neighbouring columns), in grey levels."""
    import cv2
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    if g.shape[1] > 640:
        g = cv2.resize(g, (640, max(2, g.shape[0] * 640 // g.shape[1])), interpolation=cv2.INTER_NEAREST)
    g = g.astype(np.int16)
    rows = float(np.median(np.abs(np.diff(g, axis=0)).mean(1)))
    cols = float(np.median(np.abs(np.diff(g, axis=1)).mean(0)))
    return rows, cols


def looks_garbled(img):
    """True for the 'scrambled' picture a phone sends when its camera buffer is packed with the wrong
    width: the scene repeated side by side and covered in horizontal lines. Natural pictures change
    about as much from one row to the next as from one column to the next; those change far more
    between rows. Medians are used, so ruled paper lines or text don't count."""
    rows, cols = garble_scores(img)
    return rows > 2.0 and (rows + 1) / (cols + 1) > 2.0


class _BadStream(Exception):
    pass


class MJPEGSource:
    """Reads the DroidCam phone app over Wi-Fi with urllib on a background thread that keeps only
    the newest frame, so the demo never lags behind.
    - Multipart MJPEG: each part is cut out by its Content-Length (or, without one, by the next
      boundary) and must be one complete JPEG (FFD8 ... FFD9) that cv2.imdecode accepts; anything
      else is dropped. Raw bytes are never reshaped into a picture.
    - The FIRST stream that gives real JPEGs is kept open and shown, even if its picture looks
      scrambled (then a note is shown and advice printed): the phone serves one viewer at a time,
      so hopping between URLs only makes it 'busy'. Addresses that answer 404 are never asked
      again; between real connections there is a pause of `gap` seconds.
    - Only when no stream works are single JPEG pictures polled (/cam/1/frame.jpg, /shot.jpg).
    - The first picture is saved exactly as the phone sent it (sample_path), to look at / send.
    - One 'DroidCam check' line per URL tried: URL, Content-Type, first bytes, result.
    - fps counts NEW decoded frames only. Reconnects by itself. read() returns the newest frame."""

    def __init__(self, target, timeout=5.0, start_wait=20.0, stale=3.0, verbose=True,
                 garble_check=True, sample_path=None, gap=2.5, busy_wait=12.0):
        import threading
        self.urls, self.timeout, self.stale, self.verbose = droidcam_urls(target), timeout, stale, verbose
        self.shots = droidcam_shot_urls(target) if len(self.urls) > 1 else []
        self.garble_check, self.sample_path, self.gap = garble_check, sample_path, gap
        self.url, self.good_url, self.dead = self.urls[0], None, set()
        self.frame, self.t_frame, self.count, self.bad, self.connections = None, 0.0, 0, 0, 0
        self.garbled, self.sample_saved, self._raw_first, self._t_conn = False, None, b"", 0.0
        self.times = collections.deque(maxlen=30)
        self.error, self.status, self._said, self.checks = "", "", set(), []
        self.lock, self.stop = threading.Lock(), threading.Event()
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        t0 = time.time()
        while time.time() - t0 < start_wait and self.frame is None and self.thread.is_alive():
            if self.error == "busy" and time.time() - t0 > min(busy_wait, start_wait):
                break
            time.sleep(0.05)
        if self.frame is None:
            self.release()
            raise SystemExit(self.advice() + "\n(no picture from the phone)")
        h, w = self.frame.shape[:2]
        print(f"DroidCam stream: {self.url} ({w}x{h})")

    @property
    def fps(self):
        with self.lock:
            t = list(self.times)
        if len(t) < 2 or time.time() - t[-1] > self.stale:
            return 0.0
        return (len(t) - 1) / max(1e-6, t[-1] - t[0])

    def advice(self):
        if self.error == "busy":
            return BUSY_HELP
        if self.error == "html":
            return "The phone answered with a web page instead of video.\n" + BUSY_HELP
        if self.error == "nojpeg":
            return NOJPEG_HELP
        if self.error == "notfound":
            return "The phone answered, but has no video at these addresses (HTTP 404).\n" + NOJPEG_HELP
        return NET_HELP.format(url=self.urls[0].split("/", 3)[2])

    def _say(self, key, msg):
        if self.verbose and key not in self._said:
            print(msg)
            self._said.add(key)

    def _check(self, url, ctype, first, result):
        """The one diagnostic line per URL (always printed: it is what we ask the user to paste)."""
        line = (f"DroidCam check: {url} | type={ctype or '-'} | first bytes={first[:12].hex(' ') or '-'} "
                f"| {result}")
        self.checks.append(line)
        if ("check", url, result) not in self._said:
            self._said.add(("check", url, result))
            print(line)

    def _store(self, img, url):
        with self.lock:
            self.frame, self.t_frame = img, time.time()
            self.times.append(self.t_frame)
            self.count += 1
        self.url, self.error = url, ""
        self.status = GARBLED_NOTE if self.garbled else ""
        self._said.discard("lost")

    def _save_sample(self, data):
        if not self.sample_path or self.sample_saved:
            return
        try:
            d = os.path.dirname(os.path.abspath(self.sample_path))
            os.makedirs(d, exist_ok=True)
            with open(self.sample_path, "wb") as f:
                f.write(bytes(data).rstrip(b"\r\n \t"))
            self.sample_saved = os.path.abspath(self.sample_path)
            print(f"saved the first phone picture (exactly as the phone sent it) to: {self.sample_saved}")
        except OSError as e:
            print(f"(couldn't save the sample picture to {self.sample_path}: {e})")

    def _wait_gap(self):
        """Pause between real connections so the phone has time to let go of the last one."""
        left = self._t_conn + self.gap - time.time()
        if left > 0:
            self.stop.wait(left)

    def _run(self):
        shots_tried = False
        while not self.stop.is_set():
            urls = [self.good_url] if self.good_url else [u for u in self.urls if u not in self.dead]
            for url in urls:
                if self.stop.is_set():
                    return
                if self._attempt(url) == "ok":
                    self.good_url = url                   # had pictures: always come back to this one
                    break
            else:
                if self.frame is None and not self.good_url and self.shots and not shots_tried:
                    shots_tried = True
                    if self._poll_shots():
                        return
                if not urls and not self.good_url:        # everything 404: keep trying the first slowly
                    self.dead.discard(self.urls[0])
            if self.stop.is_set():
                return
            if self.frame is not None:
                self.status = self.status or "phone stream lost: reconnecting ..."
            self.stop.wait(0.2)
            self._wait_gap()

    def _attempt(self, url):
        import socket
        import urllib.error
        self._wait_gap()
        try:
            return self._open(url)
        except urllib.error.HTTPError as e:
            try:
                body = e.read(4000)
            except Exception:
                body = b""
            ctype = e.headers.get("Content-Type", "") if e.headers else ""
            self._check(url, ctype, body, f"HTTP {e.code}" + (" (not on this phone, not asked again)" if e.code == 404 else ""))
            if e.code == 404:
                self.dead.add(url)
                if self.frame is None and self.error in ("", "network"):
                    self.error = "notfound"
                return "dead"
            self._t_conn = time.time()
            self._bad_reply(url, body, e.code)
            return "bad"
        except (urllib.error.URLError, socket.timeout, ConnectionError, OSError) as e:
            self._t_conn = time.time()
            if self.frame is None:
                self.error = "network"
            self._say(("net", str(getattr(e, "reason", e))),
                      f"can't connect to {url} ({getattr(e, 'reason', e)})\n" + NET_HELP.format(url=url.split("/", 3)[2]))
            return "net"

    def _open(self, url):
        import urllib.request
        req = urllib.request.Request(url, headers={"User-Agent": "aps360-live-demo"})
        with urllib.request.urlopen(req, timeout=self.timeout) as resp:
            self._t_conn = time.time()
            ctype = resp.headers.get("Content-Type") or ""
            low = ctype.lower()
            if "multipart" not in low and not low.startswith("image/"):
                body = resp.read(4000)
                self._check(url, ctype, body, "not a video stream")
                self._bad_reply(url, body)
                return "bad"
            self.connections += 1
            if "multipart" not in low:               # a single picture
                body = resp.read()
                img = decode_jpeg(body)
                if img is None:
                    self.error = self.error or "nojpeg"
                    self._check(url, ctype, body, "REJECTED: not a complete JPEG")
                    return "nojpeg"
                self._save_sample(body)
                self._judge(img, url, ctype, body)
                self._store(img, url)
                return "ok"
            return self._pump(resp, url, ctype)

    def _judge(self, img, url, ctype, first, n=1, k=None):
        sz = f"{img.shape[1]}x{img.shape[0]} JPEG"
        if not self.garble_check:
            self._check(url, ctype, first, f"{sz} ok (scramble check off)")
            return
        k = int(looks_garbled(img)) if k is None else k
        self.garbled = k * 2 > n
        r, c = garble_scores(img)
        sz += f" (row/col change {r:.1f}/{c:.1f})"
        if self.garbled:
            self._check(url, ctype, first, f"{sz}, {k}/{n} frames look SCRAMBLED -> showing it anyway")
            self._say("garbledhelp", GARBLED_HELP.format(sample=self.sample_saved or "droidcam_sample.jpg"))
        else:
            self._check(url, ctype, first, f"{sz} ok")

    def _parts(self, resp, ctype, limit=3_000_000):
        """Yield the payload bytes of each multipart part (cut by Content-Length, or by the next
        boundary when there is none). Bytes that aren't inside a part are thrown away; too many of
        them means this isn't a multipart picture stream at all."""
        import re
        m = re.search(r'boundary="?([^";]+)"?', ctype, re.I)
        token = m.group(1).strip().lstrip("-").encode() if m else b""
        buf, junk = b"", 0
        while not self.stop.is_set():
            chunk = resp.read1(65536) if hasattr(resp, "read1") else resp.read(16384)
            if not chunk:
                return
            if not buf and not junk and not self._raw_first:
                self._raw_first = chunk[:16]
            buf += chunk
            while True:
                h = buf.find(b"\r\n\r\n")
                if h < 0 or h > 2000:                    # no part header in sight
                    if h > 2000 or len(buf) > 4000:
                        cut = h + 4 if h > 0 else len(buf) - 3
                        junk += cut
                        buf = buf[cut:]
                        if junk > limit:
                            raise _BadStream(f"{junk // 1000} kB without any picture part")
                        continue
                    break
                head = buf[:h].decode("latin-1", "replace").lower()
                if "content-type" not in head and "content-length" not in head:
                    junk += h + 4
                    buf = buf[h + 4:]
                    if junk > limit:
                        raise _BadStream(f"{junk // 1000} kB without any picture part")
                    continue
                n = re.search(r"content-length:\s*(\d+)", head)
                start = h + 4
                if n:
                    end = start + int(n.group(1))
                    if len(buf) < end:
                        break
                    data, buf = buf[start:end], buf[end:]
                else:
                    nxt = buf.find(b"--" + token, start) if token else buf.find(b"\r\n--", start)
                    if nxt < 0:
                        if len(buf) - start > limit:
                            raise _BadStream("a part with no end")
                        break
                    data, buf = buf[start:nxt], buf[nxt:]
                yield data

    def _pump(self, resp, url, ctype):
        """Show every good JPEG of this stream until it ends. Never closes it because of how the
        picture looks: the first CHECK_FRAMES pictures only decide the note and the check line."""
        seen, good, first, k, last = 0, 0, b"", 0, None
        self._raw_first = b""
        judged = url == self.good_url and ("judged", url) in self._said
        try:
            for data in self._parts(resp, ctype):
                if not first:
                    first = data[:16] or b"?"
                seen += 1
                img = decode_jpeg(data)
                if img is None:
                    self.bad += 1
                    if good == 0 and seen >= 3 * CHECK_FRAMES:
                        self.error = self.error or "nojpeg"
                        self._check(url, ctype, first, f"REJECTED: {seen} parts, none a complete JPEG")
                        return "nojpeg"
                    continue
                good, last = good + 1, img
                if good == 1:
                    self._save_sample(data)
                if not judged:
                    if self.garble_check:
                        k += looks_garbled(img)
                    if good >= (CHECK_FRAMES if self.garble_check else 1):
                        self._judge(img, url, ctype, first, good, k)
                        judged = True
                        self._said.add(("judged", url))
                self._store(img, url)
        except _BadStream as e:
            self._check(url, ctype, first or self._raw_first, f"REJECTED: {e}")
            self.error = self.error or "nojpeg"
            return "nojpeg"
        if good and not judged:                   # stream ended before CHECK_FRAMES pictures
            self._judge(last, url, ctype, first, good, k)
            self._said.add(("judged", url))
        if not good:
            self._check(url, ctype, first or self._raw_first, f"REJECTED: {seen} parts, none a complete JPEG")
            self.error = self.error or "nojpeg"
            return "nojpeg"
        return "ok"

    def _poll_shots(self):
        """Fallback when no stream works: ask for one JPEG at a time. True if it ran (until release())."""
        import urllib.error
        import urllib.request
        for url in self.shots:
            if url in self.dead:
                continue
            if self._attempt(url) != "ok":
                continue
            print(f"using single pictures from {url} (slower than video, but readable)")
            misses = 0
            while not self.stop.is_set():
                try:
                    req = urllib.request.Request(url, headers={"User-Agent": "aps360-live-demo"})
                    with urllib.request.urlopen(req, timeout=self.timeout) as r:
                        img = decode_jpeg(r.read())
                    if img is None:
                        self.bad += 1
                    else:
                        self._store(img, url)
                        misses = 0
                        continue
                except (urllib.error.URLError, OSError):
                    misses += 1
                    self.status = "phone stream lost: reconnecting ..."
                    self.stop.wait(min(self.gap, 0.3 * misses))
            return True
        return False

    def _bad_reply(self, url, body, code=None):
        text = body.decode("utf-8", "replace").lower()
        err = "busy" if "busy" in text else "html" if ("<" in text or code) else "other"
        if self.frame is None or err == "busy":
            self.error = err
        snippet = " ".join(text.split())[:100]
        self._say(("reply", url, err), f"{url}: not a video stream ({'HTTP ' + str(code) + ', ' if code else ''}"
                                       f"'{snippet}')")
        if err == "busy":
            self._say("busyhelp", BUSY_HELP + "\n  (still trying every few seconds ...)")

    def read(self):
        with self.lock:
            f, t = self.frame, self.t_frame
        if f is None:
            return False, None
        if time.time() - t > self.stale:
            self.status = self.status if self.status and self.status != GARBLED_NOTE else "phone stream lost: reconnecting ..."
            self._say("lost", "phone stream lost: reconnecting ... (is the DroidCam app still open?)")
        return True, f.copy()

    def release(self):
        self.stop.set()
        if self.thread.is_alive() and self.thread is not __import__("threading").current_thread():
            self.thread.join(timeout=1.0)


class FileSource:
    """A picture or video file used as the 'camera' (for testing without a webcam)."""

    def __init__(self, path):
        import cv2
        self.img = cv2.imread(path) if os.path.splitext(path)[1].lower() in (".png", ".jpg", ".jpeg", ".bmp") else None
        self.cap = None if self.img is not None else cv2.VideoCapture(path)
        if self.img is None and not self.cap.isOpened():
            raise SystemExit(f"Can't open {path}")

    def read(self):
        if self.img is not None:
            return True, self.img.copy()
        ok, f = self.cap.read()
        if not ok:                                   # loop the video
            self.cap.set(1, 0)
            ok, f = self.cap.read()
        return ok, f

    def release(self):
        if self.cap is not None:
            self.cap.release()


# ------------------------------------------------------------------ drawing

def put_text(img, text, org, scale=0.7, color=(255, 255, 255), thick=2, outline=True):
    import cv2
    if outline:
        cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thick + 3, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)


def model_view(model_u8, scale=3, min_w=240):
    """The 32-px model input, enlarged (nearest neighbour) for the side window."""
    import cv2
    if model_u8 is None:
        v = np.full((32 * scale, min_w), 128, np.uint8)
        put_text(v, "no writing in the box", (8, 32 * scale // 2 + 6), 0.5)
        return v
    v = cv2.resize(model_u8, (model_u8.shape[1] * scale, model_u8.shape[0] * scale), interpolation=cv2.INTER_NEAREST)
    if v.shape[1] < min_w:
        v = np.hstack([v, np.full((v.shape[0], min_w - v.shape[1]), 200, np.uint8)])
    return v


def _header(out, keys, fps, saved):
    mode = "WORD mode" if keys.view == "word" else "PAGE mode (BETA)"
    put_text(out, f"{mode}   {fps:4.1f} FPS" + ("   FROZEN" if keys.mode != "live" else ""), (10, 28), 0.7,
             (255, 255, 255) if keys.view == "word" else (0, 220, 255))
    put_text(out, f"webcam_test: {saved} / {TARGET_WORDS} saved   (m = switch mode)", (10, 56), 0.6, (200, 255, 200))


def draw(frame, box, keys, shown, fps, saved, found_ink, note=""):
    """WORD mode overlay."""
    import cv2
    out = frame.copy()
    x0, y0, x1, y1 = box
    color = (0, 200, 0) if found_ink else (0, 200, 255)
    cv2.rectangle(out, (x0, y0), (x1, y1), color, 2)
    put_text(out, shown if shown else "(nothing read)", (x0, max(80, y0 - 12)), 1.1, (255, 255, 0), 2)
    _header(out, keys, fps, saved)
    h = out.shape[0]
    if keys.mode == "label":
        put_text(out, "Type the TRUE word, Enter = save, Esc = cancel:", (10, h - 48), 0.6, (0, 255, 255))
        put_text(out, keys.text + "_", (10, h - 16), 0.9, (0, 255, 255))
    else:
        put_text(out, "q quit | space freeze | c capture+label | a/d w/s box size | n camera | m page mode",
                 (10, h - 16), 0.5)
    if note:
        put_text(out, note, (10, h - 76), 0.6, (0, 255, 255))
    return out


def draw_page(frame, reader, keys, fps, saved, note=""):
    """PAGE mode overlay: committed words green, pending yellow; transcript in a side panel."""
    import cv2
    out = frame.copy()
    for t in reader.tracker.tracks:
        if t.missed > 0 and not t.committed:
            continue
        pts = reader.frame_box(t.box)
        cv2.polylines(out, [pts], True, (0, 200, 0) if t.committed else (0, 220, 255), 2)
    _header(out, keys, fps, saved)
    h = out.shape[0]
    put_text(out, "q quit | space freeze | r reset | s save text | c page capture | m word mode", (10, h - 16), 0.5)
    if note:
        put_text(out, note, (10, h - 48), 0.6, (0, 255, 255))
    return np.hstack([out, transcript_panel(reader, h)])


def draw_hint(img, lines, scale=0.6, top=58):
    """Warning lines on a dark-red band just under the header (above the guide box, clear of the
    key help at the bottom)."""
    import cv2
    w = img.shape[1]
    lh = int(26 * scale / 0.6)
    band = img[top:top + lh * len(lines) + 8]
    band[:] = (band * 0.25 + np.array([0, 0, 140]) * 0.75).astype(np.uint8)
    for k, line in enumerate(lines):
        s = scale
        while s > 0.3 and cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, s, 1)[0][0] > w - 20:
            s -= 0.05
        put_text(img, line, (10, top + lh * (k + 1)), s, (255, 255, 255), 1, outline=False)
    return img


def transcript_panel(reader, h, w=420):
    import textwrap
    p = np.full((h, w, 3), 250, np.uint8)
    put_text(p, "Transcript (PAGE mode, BETA)", (10, 28), 0.6, (0, 120, 255), 1, outline=False)
    n_c, n_p = len(reader.tracker.committed), len(reader.tracker.pending)
    put_text(p, f"{n_c} words | {n_p} pending", (10, 54), 0.5, (90, 90, 90), 1, outline=False)
    lines = []
    for row in reader.transcript().split("\n"):
        lines += textwrap.wrap(row, 30) or [""]
    per = max(1, (h - 80) // 26)
    for i, line in enumerate(lines[-per:]):
        put_text(p, line, (10, 84 + 26 * i), 0.65, (30, 30, 30), 1, outline=False)
    return p


def save_transcript(reader, folder):
    os.makedirs(folder, exist_ok=True)
    path = os.path.join(folder, time.strftime("page_%Y%m%d_%H%M%S.txt"))
    with open(path, "w", encoding="utf-8") as f:
        f.write(reader.transcript() + "\n")
    return path


def page_capture(reader, out_dir, cam_name, frame_shape):
    """Save every word crop of the last processed page with the model's guess as label (needs_check=1)."""
    n = 0
    for _, crop_bgr, guess in reader.last_crops:
        if crop_bgr.size and guess:
            save_capture(out_dir, crop_bgr, guess, guess, cam_name + " page", frame_shape, needs_check=True)
            n += 1
    return n, count_saved(out_dir)


# ------------------------------------------------------------------ main loop

def parse_args(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--camera", type=int, default=None, help="camera index (default: first that works of 0-5)")
    ap.add_argument("--list", action="store_true", help="list cameras (name, resolution, live or placeholder) and quit")
    ap.add_argument("--device-name", default=None,
                    help="prefer the camera whose Windows name contains this text (any case), e.g. DroidCam "
                         "or part of your phone's name; names containing 'DroidCam' are preferred anyway")
    ap.add_argument("--model", default=os.path.join(HERE, "crnn_model.pt"))
    ap.add_argument("--out", default=os.path.join(HERE, "webcam_test"), help="where captures are saved")
    ap.add_argument("--transcripts", default=os.path.join(HERE, "transcripts"), help="where page transcripts go")
    ap.add_argument("--every", type=int, default=3, help="WORD mode: run the model every N frames")
    ap.add_argument("--page-interval", type=float, default=0.5, help="PAGE mode: seconds between page readings")
    ap.add_argument("--smooth", type=int, default=5, help="majority vote over the last N readings")
    ap.add_argument("--box", type=float, nargs=2, default=(0.7, 0.25), metavar=("W", "H"),
                    help="guide box size as a fraction of the frame")
    ap.add_argument("--rotate", type=int, default=0, choices=(0, 90, 180, 270), help="rotate the camera picture")
    ap.add_argument("--threads", type=int, default=min(4, os.cpu_count() or 1))
    ap.add_argument("--page", action="store_true", help="start in PAGE mode (beta)")
    ap.add_argument("--droidcam", metavar="IP[:PORT]",
                    help="read the DroidCam phone app directly over Wi-Fi (no DroidCam Client needed), "
                         "e.g. --droidcam 10.0.0.74 (port 4747 by default) or a full http:// URL")
    ap.add_argument("--save-frame", default=os.path.join(HERE, "droidcam_sample.jpg"), metavar="PATH",
                    help="--droidcam: save the first phone picture, exactly as the phone sent it, here "
                         "(default: droidcam_sample.jpg next to live_demo.py; '' = don't save)")
    ap.add_argument("--no-garble-check", action="store_true",
                    help="--droidcam: don't check whether the phone's picture looks scrambled")
    ap.add_argument("--backend", default="auto", choices=("auto", "dshow", "msmf", "any"),
                    help="OpenCV camera API (auto = DirectShow, then Media Foundation, then any)")
    ap.add_argument("--image", help="use a picture or video file (or an http:// MJPEG stream) instead of a camera")
    ap.add_argument("--headless", action="store_true", help="no windows: print predictions (testing)")
    ap.add_argument("--max-frames", type=int, default=0, help="stop after N frames (0 = run until q)")
    ap.add_argument("--capture-label", help="headless testing: save the last crop with this label")
    ap.add_argument("--keys", default="", help="headless testing: one key per frame ('.' = no key)")
    ap.add_argument("--no-pause", action="store_true", help="don't wait for Enter after an error")
    return ap.parse_args(argv)


def main(argv=None):
    a = parse_args(argv)
    import cv2
    from src.demo.page_reader import PageReader
    from src.demo.webcam import Recognizer, crop, guide_box
    global BACKEND
    BACKEND = a.backend
    names, indexed = camera_names()
    by_index = indexed_names(names, indexed)
    if a.list:
        print(f"checking cameras 0-{MAX_CAMERAS - 1} (about 1 s each) ...")
        cams = describe_cameras()
        for i, w, h, verdict in cams:
            label = f' "{by_index[i]}"' if i in by_index else ""
            print(f"  camera {i}{label}: {w}x{h}, {verdict}")
        if not cams:
            print("  no camera gives a picture (is the DroidCam Client started?)")
        if names and not indexed:
            print("camera names Windows reports (their numbers are unknown; `python -m pip install pygrabber` "
                  "shows which number is which):")
            for nm in names:
                print("   -", nm)
        elif not names:
            print("(camera names not available)")
        print("use one with:  python live_demo.py --camera N   or   python live_demo.py --device-name DroidCam")
        return cams
    if not os.path.exists(a.model):
        raise SystemExit(f"Model file not found: {a.model}\nKeep live_demo.py next to crnn_model.pt (unzip the whole folder).")
    rec = Recognizer(a.model, threads=a.threads)
    reader = PageReader(rec)
    print(f"model loaded ({sum(p.numel() for p in rec.model.parameters()):,} parameters, CPU)")

    cams = []
    stream = a.droidcam or (a.image if is_stream_url(a.image) else None)
    is_cam = not (a.image or a.droidcam)
    if stream:
        print(f"connecting to the phone: {droidcam_urls(stream)[0].split('/', 3)[2]} ...")
        src = MJPEGSource(stream, garble_check=not a.no_garble_check, sample_path=a.save_frame or None)
        cam_name = "droidcam " + droidcam_urls(stream)[0].split("/", 3)[2]
    elif a.image:
        src, cam_name = FileSource(a.image), os.path.basename(a.image)
    else:
        if a.camera is None:
            if by_index:
                print("cameras:", ", ".join(f'{i} "{n}"' for i, n in sorted(by_index.items())))
            if a.device_name and by_index and not any(name_matches(n, a.device_name) for n in by_index.values()):
                print(f"no camera name contains '{a.device_name}'; trying the others")
            order = preferred_order(range(MAX_CAMERAS), by_index, a.device_name)
            print(f"looking for cameras 0-{MAX_CAMERAS - 1} ...")
            cams = [c[0] for c in probe_cameras(order)]
            if not cams:
                raise SystemExit("No camera found. Is the DroidCam Client connected (it must say 'connected' "
                                 "and show the phone picture)? Then run again, or try --camera 0 / 1 / 2.")
            a.camera = cams[0]
            why = f' ("{by_index[a.camera]}")' if a.camera in by_index else ""
            print(f"working cameras: {sorted(cams)} -> using {a.camera}{why} "
                  "(press n in the window to switch, or use --camera N)")
        else:
            cams = [a.camera]
        src = open_camera(a.camera)
        if src is None:
            raise SystemExit(f"Camera {a.camera} doesn't give a picture. Try another: python live_demo.py --list")
        cam_name = f"camera {a.camera}"
        print(f"using camera {a.camera}" + (f' "{by_index[a.camera]}"' if a.camera in by_index else ""))

    keys, smooth = Keys("page" if a.page else "word"), Smoother(a.smooth)
    wfrac, hfrac = a.box
    saved = count_saved(a.out)
    frozen, last_crop, raw, shown, model_u8, found = None, None, "", "", None, False
    fps, t_prev, n, note, note_until, t_page = 0.0, time.time(), 0, "", 0, 0.0
    interval = 0.0 if a.headless else a.page_interval
    still, warned = StillDetector(), False
    rot = {90: cv2.ROTATE_90_CLOCKWISE, 180: cv2.ROTATE_180, 270: cv2.ROTATE_90_COUNTERCLOCKWISE}.get(a.rotate)
    if not a.headless:
        cv2.namedWindow("Live demo", cv2.WINDOW_NORMAL)
        cv2.namedWindow("What the model sees", cv2.WINDOW_AUTOSIZE)
        cv2.moveWindow("What the model sees", 20, 20)

    def read_word(frame, box):
        c = crop(frame, box)
        t, mu, g = rec.read(c)
        return c, t, mu, g is not None

    try:
        while True:
            if keys.mode == "live" or frozen is None:
                ok, frame = src.read()
                if (not ok or frame is None) and is_cam:
                    print(f"camera {a.camera} stopped giving frames: re-opening it ...")
                    try:
                        src.release()
                    except Exception:
                        pass
                    src = _open_or_none(a.camera) or BlankSource()
                    ok, frame = src.read()
                if not ok or frame is None:
                    print("the picture source stopped giving frames"); break
                if is_cam and still.add(frame) and not warned:
                    print("WARNING: " + " ".join(PLACEHOLDER_HINT) + " (or press n for the next camera)")
                    warned = True
                if rot is not None:
                    frame = cv2.rotate(frame, rot)
                frozen = frame if keys.mode != "live" else None
            else:
                frame = frozen
            box = guide_box(frame.shape, wfrac, hfrac)
            n += 1
            now = time.time()
            if keys.view == "word":
                if keys.mode == "live" and (n - 1) % max(1, a.every) == 0:
                    last_crop, raw, model_u8, found = read_word(frame, box)
                    shown = smooth.add(raw if found else "")
                    if a.headless:
                        print(f"frame {n}: '{shown}' (this frame '{raw}')")
                elif keys.mode != "live" and last_crop is None:
                    last_crop, raw, model_u8, found = read_word(frame, box); shown = raw
            elif keys.mode == "live" and now - t_page >= interval:
                reader.process(frame); t_page = now
                model_u8 = reader.last_model_input
                if a.headless:
                    print(f"frame {n}: {len(reader.tracker.committed)} words committed, "
                          f"{len(reader.tracker.pending)} pending")
            if stream:                                     # count NEW phone pictures, not loop turns
                fps = src.fps
            else:
                fps = 0.9 * fps + 0.1 / max(1e-6, now - t_prev) if fps else 1 / max(1e-6, now - t_prev)
            t_prev = now
            if a.headless:
                if a.max_frames and n >= a.max_frames:
                    break
                k = a.keys[n - 1] if n - 1 < len(a.keys) else "."
                act = None if k == "." else keys.handle(ord(k))
            else:
                msg = note if now < note_until else ""
                view = draw(frame, box, keys, shown, fps, saved, found, msg) if keys.view == "word" else \
                    draw_page(frame, reader, keys, fps, saved, msg)
                if still.looks_placeholder and is_cam:
                    draw_hint(view, PLACEHOLDER_HINT + ("(or press n to try the next camera)",))
                elif getattr(src, "status", ""):
                    draw_hint(view, (src.status,))
                cv2.imshow("Live demo", view)
                cv2.imshow("What the model sees", model_view(model_u8))
                act = keys.handle(cv2.waitKey(1))
                if cv2.getWindowProperty("Live demo", cv2.WND_PROP_VISIBLE) < 1:
                    break
            if act == "quit":
                break
            if act in ("freeze", "label"):
                frozen = frame
                if keys.view == "word":                  # read the frozen word once, without smoothing
                    last_crop, raw, model_u8, found = read_word(frame, box); shown = raw
                else:
                    reader.process(frame)
            elif act == "unfreeze" or act == "mode":
                smooth.clear(); last_crop = None
                if act == "mode":
                    note, note_until = ("PAGE mode is BETA: less accurate than WORD mode" if keys.view == "page"
                                        else "WORD mode"), now + 3
            elif act == "save":
                rel, saved = save_capture(a.out, last_crop, keys.text, raw, cam_name, frame.shape)
                note, note_until = f"saved {rel} as '{keys.text.strip()}'", now + 2.5
                print(f"saved {rel}: '{keys.text.strip()}' (model said '{raw}') -> {saved} words")
                keys.saved(); smooth.clear()
            elif act == "empty":
                note, note_until = "type the word first (Esc cancels)", now + 2
            elif act == "reset":
                reader.reset(); note, note_until = "transcript cleared", now + 2
            elif act == "save_text":
                p = save_transcript(reader, a.transcripts)
                note, note_until = f"saved {os.path.basename(p)}", now + 3
                print("transcript saved to", p)
            elif act == "page_capture":
                frozen = frame
                reader.process(frame, force_read=True)
                k, saved = page_capture(reader, a.out, cam_name, frame.shape)
                note, note_until = f"saved {k} word crops with GUESSED labels: fix them in labels.csv", now + 5
                print(f"page capture: {k} crops saved with the model's guesses as labels (needs_check=1). "
                      f"Edit {os.path.join(a.out, LABELS_CSV)}: correct each label and set needs_check to 0.")
            elif act in ("narrower", "wider"):
                wfrac = float(np.clip(wfrac + (0.05 if act == "wider" else -0.05), 0.2, 0.95))
            elif act in ("taller", "shorter"):
                hfrac = float(np.clip(hfrac + (0.03 if act == "taller" else -0.03), 0.08, 0.6))
            elif act == "next_camera" and not is_cam:
                note, note_until = ("reading the phone over Wi-Fi: n does nothing here" if stream else
                                    "reading a file: n does nothing here"), time.time() + 2
            elif act == "next_camera":
                print(f"n pressed: looking for another camera than {a.camera} (0-{MAX_CAMERAS - 1}) ...")
                src, a.camera, msg = switch_camera(src, a.camera, names=by_index)
                cam_name = f"camera {a.camera}"; smooth.clear(); reader.reset(); still.reset()
                warned, frozen, last_crop = False, None, None
                keys.mode = "live"
                note, note_until = msg, time.time() + 3
    finally:
        if src is not None:
            try:
                src.release()
            except Exception:
                pass
        if not a.headless:
            cv2.destroyAllWindows()
    if keys.view == "page":
        print("TRANSCRIPT:\n" + reader.transcript())
        return reader.transcript()
    if a.headless and a.capture_label and last_crop is not None:
        rel, saved = save_capture(a.out, last_crop, a.capture_label, raw, cam_name, frame.shape)
        print(f"saved {rel} -> {saved} words")
    print(f"FINAL: '{shown}'")
    return shown


def run(argv=None):
    """main(), but an error never just closes the window: the full traceback is printed and the
    program waits for Enter, so it stays readable (e.g. in the VS Code terminal)."""
    no_pause = "--no-pause" in (sys.argv[1:] if argv is None else argv)

    def wait():
        if not no_pause and sys.stdin is not None and sys.stdin.isatty():
            try:
                input("\nPress Enter to close ...")
            except EOFError:
                pass
    try:
        return main(argv)
    except KeyboardInterrupt:
        print("stopped (Ctrl+C)")
    except SystemExit as e:
        if e.code not in (None, 0):
            if isinstance(e.code, str):
                print(e.code)
            wait()
            sys.exit(1)
        raise
    except Exception:
        print("\n*** live_demo.py crashed. Please copy everything below when asking for help: ***\n")
        traceback.print_exc()
        wait()
        sys.exit(1)


if __name__ == "__main__":
    run()

In [ ]:
%%writefile /content/aps360_htr/src/demo/page_reader.py
"""PAGE mode (beta) of the live demo: a running transcript of a whole page.

WORD mode (one word in the guide box) is the reliable mode; page mode adds three error sources
(finding the paper, cutting it into words, tracking words between frames), so it is less accurate.

Pipeline, about twice a second:
  1. PageSegmenter: find the paper (largest bright 4-corner shape) and straighten it with a
     perspective warp (if no paper outline is found, the whole frame is the page); flatten the
     lighting and binarize (webcam.flatten_lighting + Otsu); remove ruled lines; split into
     lines with the horizontal projection (one peak per line, cut at the emptiest row between
     peaks); connected components are given to the line their centre falls in, then joined into
     words left to right when the gap is below `word_gap` x the typical letter height;
     heuristics drop specks, drawings and page edges.
  2. Each word crop is read by the CRNN with exactly the word-mode preprocessing
     (webcam.clean_photo -> training transform -> greedy CTC).
  3. WordTracker de-duplicates by POSITION, never by text: boxes live in paper coordinates
     (straightened page; remaining drift is removed by registering each frame's boxes to the
     tracked boxes with the median offset, plus an IoU/overlap tolerance);
     a box is committed only after it has been stable for `commit_after` updates with a
     consistent majority-vote reading; a committed box is never added twice (its text may be
     refined); ink growing at the right edge of a word updates that word instead of adding one.
  4. Transcript: committed words ordered by line, then by x.
"""
import collections
from dataclasses import dataclass, field

import numpy as np

from src.demo.webcam import clean_photo, flatten_lighting


def _cv2():
    import cv2
    return cv2


# ------------------------------------------------------------------ geometry helpers

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def overlap_small(a, b):
    """Intersection / area of the smaller box (1.0 when one box contains the other)."""
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    m = min((a[2] - a[0]) * (a[3] - a[1]), (b[2] - b[0]) * (b[3] - b[1]))
    return ix * iy / m if m > 0 else 0.0


def shift_box(b, dx, dy):
    return (b[0] + dx, b[1] + dy, b[2] + dx, b[3] + dy)


# ------------------------------------------------------------------ 1. segmentation

@dataclass
class SegResult:
    page: np.ndarray                 # straightened, lighting-flattened gray page (uint8)
    color: np.ndarray                # straightened colour page (for saving crops)
    boxes: list                      # word boxes (x0, y0, x1, y1) in page coordinates
    lines: list                      # line index for every box
    to_frame: object = None          # 3x3 matrix page -> frame (None = identity)
    text_height: float = 0.0
    paper_found: bool = False


class PageSegmenter:
    def __init__(self, page_width=1000, straighten=True, word_gap=0.45, min_paper_frac=0.2):
        self.page_width, self.straighten = page_width, straighten
        self.word_gap = word_gap                 # letters closer than word_gap x text height are one word
        self.min_paper_frac = min_paper_frac

    # -- paper
    def find_paper(self, gray):
        """4 corners (tl, tr, br, bl) of the paper, or None."""
        cv2 = _cv2()
        h, w = gray.shape
        s = 400 / max(h, w)
        small = cv2.GaussianBlur(cv2.resize(gray, (int(w * s), int(h * s))), (5, 5), 0)
        _, bright = cv2.threshold(small, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        bright = cv2.morphologyEx(bright, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
        cs, _ = cv2.findContours(bright, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cs:
            return None
        c = max(cs, key=cv2.contourArea)
        area = cv2.contourArea(c)
        if area < self.min_paper_frac * small.size or area > 0.97 * small.size:
            return None                           # no outline (paper fills the frame) or too small
        approx = cv2.approxPolyDP(c, 0.03 * cv2.arcLength(c, True), True)
        if len(approx) != 4 or not cv2.isContourConvex(approx):
            return None
        p = approx.reshape(4, 2).astype(np.float32) / s
        ssum, diff = p.sum(1), np.diff(p, axis=1).ravel()
        return np.float32([p[ssum.argmin()], p[diff.argmin()], p[ssum.argmax()], p[diff.argmax()]])

    def warp(self, frame, gray):
        cv2 = _cv2()
        corners = self.find_paper(gray) if self.straighten else None
        if corners is None:
            return frame, gray, None, False
        tl, tr, br, bl = corners
        pw = max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl))
        ph = max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr))
        W = self.page_width; H = int(W * ph / max(pw, 1))
        dst = np.float32([[0, 0], [W - 1, 0], [W - 1, H - 1], [0, H - 1]])
        M = cv2.getPerspectiveTransform(corners, dst)
        inset = lambda im: cv2.warpPerspective(im, M, (W, H), borderMode=cv2.BORDER_REPLICATE)
        return inset(frame), inset(gray), np.linalg.inv(M), True

    # -- ink and words
    def ink(self, page_gray):
        cv2 = _cv2()
        flat = flatten_lighting(page_gray)
        u8 = flat.astype(np.uint8)
        t, _ = cv2.threshold(cv2.GaussianBlur(u8, (3, 3), 0), 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        t = min(t, 200)                           # an almost-empty page: Otsu would pick paper noise
        m = (u8 < t).astype(np.uint8)
        h, w = m.shape
        lines = cv2.morphologyEx(m, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (max(15, w // 6), 1)))
        m[lines > 0] = 0                          # ruled lines / page edges
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
        e = 2 + int(0.008 * min(h, w))            # page border: leftovers of the desk / paper edge
        m[:e] = 0; m[-e:] = 0; m[:, :e] = 0; m[:, -e:] = 0
        return flat.astype(np.uint8), m

    def segment(self, frame):
        cv2 = _cv2()
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) if frame.ndim == 3 else frame
        color, pgray, to_frame, found = self.warp(frame, gray)
        flat, m = self.ink(pgray)
        H, W = m.shape
        n, lab, st, cen = cv2.connectedComponentsWithStats(m, connectivity=8)
        comps = [(int(x), int(y), int(x + w), int(y + h), int(a), float(cy))
                 for (x, y, w, h, a), (_, cy) in zip(st[1:], cen[1:]) if a >= 4]
        big = [c for c in comps if c[4] >= max(12, 2e-5 * m.size)]
        if not big:
            return SegResult(flat, color, [], [], to_frame, 0.0, found)
        areas = np.array([c[4] for c in big])
        th = float(np.median([c[3] - c[1] for c in big if c[4] >= np.percentile(areas, 30)]))
        comps = [c for c in comps if (c[3] - c[1]) < 5 * th and (c[2] - c[0]) < 0.95 * W]   # drop drawings/edges
        bands = self._line_bands(m, th)
        rows = [[] for _ in bands]
        for c in comps:
            li = int(np.searchsorted([b1 for _, b1 in bands], c[5]))
            if li < len(bands):
                rows[li].append(c)
        boxes, lines = [], []
        k = self.word_gap * th
        for li, row in enumerate(rows):
            words = []
            for c in sorted(row):
                if words and c[0] - words[-1][2] < k:          # small gap (or overlap): same word
                    w = words[-1]
                    words[-1] = [min(w[0], c[0]), min(w[1], c[1]), max(w[2], c[2]), max(w[3], c[3]), w[4] + c[4]]
                else:
                    words.append(list(c[:5]))
            for w in words:
                if w[4] >= max(12, 0.05 * th * th) and (w[3] - w[1]) >= 0.25 * th:   # not just a speck
                    boxes.append(tuple(w[:4])); lines.append(li)
        # renumber lines that ended up empty
        used = sorted(set(lines)); ren = {l: i for i, l in enumerate(used)}
        lines = [ren[l] for l in lines]
        return SegResult(flat, color, boxes, lines, to_frame, th, found)

    def _line_bands(self, m, th):
        """[(y_start, y_end)] per text line from the horizontal projection: smoothed ink per row,
        one peak per line (peaks at least ~0.9 text heights apart), cut at the emptiest row between
        two peaks. Touching ascenders/descenders are fine: a shallow valley still separates lines."""
        cv2 = _cv2()
        prof = m.sum(1).astype(np.float32)
        sig = max(1.0, 0.3 * th)
        prof = cv2.GaussianBlur(prof.reshape(-1, 1), (1, 0), sig).ravel() if len(prof) > 3 else prof
        if prof.max() <= 0:
            return []
        cand = [y for y in range(1, len(prof) - 1) if prof[y] >= prof[y - 1] and prof[y] >= prof[y + 1]
                and prof[y] > 0.12 * prof.max()]
        peaks = []
        for y in sorted(cand, key=lambda y: -prof[y]):            # strongest first, keep them apart
            if all(abs(y - q) > 0.9 * th for q in peaks):
                peaks.append(y)
        peaks.sort()
        merged = [peaks[0]]
        for q in peaks[1:]:                                       # no real valley in between: same line
            v = prof[merged[-1]:q + 1].min()
            if v > 0.75 * min(prof[merged[-1]], prof[q]):
                merged[-1] = merged[-1] if prof[merged[-1]] >= prof[q] else q
            else:
                merged.append(q)
        cuts = [0] + [int(p + np.argmin(prof[p:q + 1])) for p, q in zip(merged, merged[1:])] + [len(prof)]
        return list(zip(cuts[:-1], cuts[1:]))


# ------------------------------------------------------------------ 3. tracking / de-duplication

@dataclass
class Track:
    id: int
    box: tuple
    votes: collections.deque = field(default_factory=lambda: collections.deque(maxlen=7))
    stable: int = 1
    missed: int = 0
    committed: bool = False
    text: str = ""
    age: int = 0
    last_read: int = -99

    def majority(self):
        if not self.votes:
            return "", 0.0
        t, n = collections.Counter(self.votes).most_common(1)[0]
        return t, n / len(self.votes)


class WordTracker:
    """Matches word boxes between frames by position and commits each word exactly once."""

    def __init__(self, commit_after=3, agree=0.6, iou_same=0.55, forget_pending=4):
        self.commit_after, self.agree, self.iou_same, self.forget = commit_after, agree, iou_same, forget_pending
        self.tracks, self.next_id, self.step = [], 1, 0

    def reset(self):
        self.tracks, self.next_id, self.step = [], 1, 0

    def match(self, boxes):
        """For each box: the index of the track it belongs to, or None (new word), or -1 (ignore:
        the box covers two or more committed words, e.g. two words merged by the splitter)."""
        out, used = [], set()
        for b in boxes:
            cands = [(iou(b, t.box), overlap_small(b, t.box), i) for i, t in enumerate(self.tracks)]
            covering = [i for s, o, i in cands if o > 0.7 and self.tracks[i].committed]
            if len(covering) >= 2:
                out.append(-1); continue
            good = [(s, o, i) for s, o, i in cands if (s > 0.3 or o > 0.7) and i not in used]
            if not good:
                out.append(None); continue
            s, o, i = max(good)
            used.add(i); out.append(i)
        return out

    def needs_read(self, idx, box):
        """Committed words whose box hasn't changed are re-read only every 4th update (CPU time)."""
        if idx is None or idx < 0:
            return idx is None
        t = self.tracks[idx]
        return not (t.committed and iou(box, t.box) > 0.85 and self.step - t.last_read < 4)

    def update(self, boxes, texts, matches=None):
        """boxes: detections in reference-paper coordinates; texts: CRNN reading per box (None = not
        re-read this time). Returns the ids of newly committed tracks."""
        self.step += 1
        matches = self.match(boxes) if matches is None else matches
        seen, new_commits = set(), []
        for b, txt, i in zip(boxes, texts, matches):
            if i == -1:
                continue
            if i is None:
                t = Track(self.next_id, tuple(b)); self.next_id += 1
                if txt is not None:
                    t.votes.append(txt); t.last_read = self.step
                self.tracks.append(t); seen.add(t.id)
                continue
            t = self.tracks[i]; seen.add(t.id)
            same = iou(b, t.box) > self.iou_same
            h = t.box[3] - t.box[1]
            grew_right = ((b[2] - t.box[2]) > 0.15 * h and abs(b[0] - t.box[0]) < 0.4 * h
                          and overlap_small(b, t.box) > 0.85)          # the old box is (almost) inside the new one
            if grew_right:                        # the word is still being written: same word, new box
                t.box, t.stable = tuple(b), 1
                t.votes.clear()
            elif same:
                t.stable += 1
                t.box = tuple(int(round(0.7 * u + 0.3 * v)) for u, v in zip(t.box, b))   # smooth jitter
            elif not t.committed:                 # a different box (e.g. a bad cut last time): start over
                t.box, t.stable = tuple(b), 1
                t.votes.clear()
            else:                                 # committed word seen with a different box: keep it as is
                t.stable = 1
            if txt is not None:
                t.votes.append(txt); t.last_read = self.step
            t.missed = 0
            text, frac = t.majority()
            if not t.committed and t.stable >= self.commit_after and text and frac >= self.agree:
                t.committed, t.text = True, text
                new_commits.append(t.id)
            elif t.committed and text and frac >= self.agree and same:
                t.text = text                     # refine, never duplicate
        keep = []
        for t in self.tracks:
            if t.id not in seen:
                t.missed += 1
            t.age += 1
            if t.committed or t.missed <= self.forget:
                keep.append(t)                    # committed words stay even when out of view
        self.tracks = keep
        return new_commits

    @property
    def committed(self):
        return [t for t in self.tracks if t.committed]

    @property
    def pending(self):
        return [t for t in self.tracks if not t.committed]


class Transcript:
    """Committed words ordered by line (vertical centre), then by x."""

    @staticmethod
    def lines(tracks):
        ts = sorted(tracks, key=lambda t: (t.box[1] + t.box[3]) / 2)
        if not ts:
            return []
        h = float(np.median([t.box[3] - t.box[1] for t in ts]))
        rows, centers = [], []
        for t in ts:
            c = (t.box[1] + t.box[3]) / 2
            if centers and abs(c - np.mean(centers[-1])) < 0.5 * h:
                rows[-1].append(t); centers[-1].append(c)
            else:
                rows.append([t]); centers.append([c])
        return [sorted(r, key=lambda t: t.box[0]) for r in rows]

    @classmethod
    def text(cls, tracks):
        return "\n".join(" ".join(t.text for t in row) for row in cls.lines(tracks))


# ------------------------------------------------------------------ everything together

class PageReader:
    def __init__(self, recognizer, segmenter=None, tracker=None, align=True):
        self.rec = recognizer
        self.seg = segmenter or PageSegmenter()
        self.tracker = tracker or WordTracker()
        self.align = align
        self.shift = (0.0, 0.0)                    # page coords -> reference (track) coords
        self.last = None                           # last SegResult
        self.last_crops = []                       # (box in page coords, colour crop, reading) of the last frame
        self.last_model_input = None

    def reset(self):
        self.tracker.reset(); self.shift = (0.0, 0.0)

    def _register(self, boxes):
        """Shift (dx, dy) from this frame's page coordinates to the reference (track) coordinates:
        start from the last shift, pair every detection with the track it overlaps most, and move by
        the median offset of those pairs (twice). Robust to new words appearing (unlike image
        correlation) and to a few wrong pairs; follows the paper drifting a little every frame."""
        dx, dy = self.shift
        tracks = [t for t in self.tracker.tracks if t.committed or t.stable >= 2]
        if not tracks or not boxes:
            return dx, dy
        c = lambda b: ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2)
        for _ in range(2):
            offs = []
            for b in boxes:
                sb = shift_box(b, dx, dy)
                best = max(tracks, key=lambda t: iou(sb, t.box))
                if iou(sb, best.box) > 0.3:
                    (tx, ty), (bx, by) = c(best.box), c(sb)
                    offs.append((tx - bx, ty - by))
            if not offs:
                break
            mx, my = np.median(offs, axis=0)
            dx, dy = dx + mx, dy + my
        return dx, dy

    def process(self, frame, force_read=False):
        """Read one frame. force_read=True re-reads every word (page capture)."""
        from src.demo.webcam import tensor_to_u8
        r = self.seg.segment(frame)
        self.last = r
        self.shift = self._register(r.boxes) if self.align else (0.0, 0.0)
        dx, dy = self.shift
        ref_boxes = [tuple(int(round(v)) for v in shift_box(b, dx, dy)) for b in r.boxes]
        matches = self.tracker.match(ref_boxes)
        texts, crops = [], []
        for b, rb, mi in zip(r.boxes, ref_boxes, matches):
            pad = int(0.15 * (b[3] - b[1])) + 2
            y0, y1 = max(0, b[1] - pad), min(r.page.shape[0], b[3] + pad)
            x0, x1 = max(0, b[0] - pad), min(r.page.shape[1], b[2] + pad)
            txt = None
            if force_read or self.tracker.needs_read(mi, rb):
                g = clean_photo(r.page[y0:y1, x0:x1])
                if g is not None:
                    x = self.rec.tensor(g)
                    txt = self.rec.predict_tensor(x)
                    self.last_model_input = tensor_to_u8(x)
                else:
                    txt = ""
            texts.append(txt)
            crops.append(((x0, y0, x1, y1), r.color[y0:y1, x0:x1].copy(), txt))
        self.last_crops = crops
        self.tracker.update(ref_boxes, texts, matches)
        return r

    def transcript(self):
        return Transcript.text(self.tracker.committed)

    def frame_box(self, ref_box):
        """A reference-paper box -> 4 frame points (for drawing)."""
        dx, dy = self.shift
        x0, y0, x1, y1 = shift_box(ref_box, -dx, -dy)
        pts = np.float32([[x0, y0], [x1, y0], [x1, y1], [x0, y1]])
        M = self.last.to_frame if self.last is not None else None
        if M is not None:
            ph = np.hstack([pts, np.ones((4, 1), np.float32)]) @ M.T
            pts = ph[:, :2] / ph[:, 2:3]
        return pts.astype(np.int32)

### Building the live_demo folder, scoring, report outputs

In [ ]:
%%writefile /content/aps360_htr/src/demo/bundle.py
"""Build the self-contained live_demo folder (+ live_demo.zip) for the Windows laptop.

Everything the demo imports is copied unchanged from this codebase, so the laptop runs exactly the
model, charset and preprocessing code used in training and evaluation.
"""
import json
import os
import shutil
import zipfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
BUNDLED = ["src/__init__.py", "src/config.py", "src/data/__init__.py", "src/data/charset.py",
           "src/data/transforms.py", "src/models/__init__.py", "src/models/crnn.py",
           "src/demo/__init__.py", "src/demo/webcam.py", "src/demo/page_reader.py"]
REQUIREMENTS = """# CPU-only is enough. On Windows, the normal 'torch' from pip is the CPU build.
torch>=2.1
opencv-python>=4.8
numpy>=1.24
pillow>=10.0
# optional, Windows only: lets the demo show camera NAMES (e.g. "DroidCam Video") next to their numbers
pygrabber>=0.2; sys_platform == "win32"
"""


def build(out_dir, model_file, info=None, samples=()):
    """Write live_demo/ into out_dir. `samples` = [(png path, label)] for testing without a camera.
    Only our own files are (over)written; anything else in the folder (e.g. webcam_test/) is kept."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    for rel in BUNDLED:
        (out / rel).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(ROOT / rel, out / rel)
    shutil.copy(ROOT / "src/demo/live_demo.py", out / "live_demo.py")
    shutil.copy(ROOT / "src/demo/README_live_demo.md", out / "README.md")
    (out / "requirements.txt").write_text(REQUIREMENTS)
    if Path(model_file).resolve() != (out / "crnn_model.pt").resolve():
        shutil.copy(model_file, out / "crnn_model.pt")
    (out / "model_info.json").write_text(json.dumps(info or {}, indent=2))
    if samples:
        (out / "samples").mkdir(exist_ok=True)
        rows = ["file,label"]
        for p, label in samples:
            shutil.copy(p, out / "samples" / Path(p).name)
            rows.append(f"{Path(p).name},{label}")
        (out / "samples" / "labels.csv").write_text("\n".join(rows) + "\n")
    return sorted(str(p.relative_to(out)) for p in out.rglob("*") if p.is_file() and "__pycache__" not in p.parts)


def make_zip(folder, zip_path, skip=("webcam_test", "__pycache__")):
    """live_demo.zip with a top-level live_demo/ folder (captures in webcam_test/ are left out)."""
    folder = Path(folder)
    tmp = str(zip_path) + ".tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(folder.rglob("*")):
            rel = p.relative_to(folder)
            if p.is_file() and not (set(rel.parts) & set(skip)):
                z.write(p, f"live_demo/{rel.as_posix()}")
    os.replace(tmp, zip_path)
    return str(zip_path)

In [ ]:
%%writefile /content/aps360_htr/src/demo/score.py
"""Score the webcam test set (captured with live_demo.py) with every model.

All models read the SAME cleaned image of each word (webcam.clean_photo, as in the live demo):
  - CRNN: the training transform + greedy CTC, exactly as in the demo;
  - Tesseract: the same settings as Step 3 (white 16-px border, --oem 1 --psm 8);
  - EMNIST baseline: the same splitter settings and IAM prior that Step 4 chose on validation.
Rows marked needs_check=1 (PAGE-mode page captures, labelled with the model's guess) are skipped
until the user has corrected the label and set needs_check to 0.
Two CRNNs are scored: Model A (live_demo/crnn_model.pt, trained on clean IAM) and Model B
(outputs/model_b/crnn_model_b.pt, same architecture and settings, trained with camera-style augmentation);
architecture_diff / config_diff check that Model B really has Model A's architecture before scoring.
An extra CRNN row without the photo clean-up (raw guide-box crop) shows what the clean-up is worth.
Labels are compared case-sensitively with the same CER/WER code as everywhere else.
"""
import csv
import os
import time

import numpy as np

from src.demo.webcam import clean_photo, to_gray
from src.utils.metrics import cer, wer


def find_webcam_dirs(project_dir, max_depth=3):
    """Folders called webcam_test* (with a labels.csv) under the project folder, most rows first."""
    hits = []
    base = os.path.abspath(project_dir)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth >= max_depth:
            dirs[:] = []
        if os.path.basename(root).startswith("webcam_test") and "labels.csv" in files:
            hits.append((len(read_labels(root)), root))
    return [r for _, r in sorted(hits, key=lambda t: -t[0])]


def read_labels(folder, unchecked=None):
    """[{'file', 'label', ...}] for rows whose image exists (paths in labels.csv are relative).
    Rows with needs_check=1 (PAGE-mode captures whose label is only the model's guess) are left out
    and their files appended to `unchecked`: a guess is not a true label."""
    p = os.path.join(folder, "labels.csv")
    if not os.path.exists(p):
        return []
    with open(p, newline="", encoding="utf-8-sig") as f:
        rows = [r for r in csv.DictReader(f) if (r.get("label") or "").strip() and r.get("file")]
    out = []
    for r in rows:
        if str(r.get("needs_check") or "0").strip() not in ("", "0"):
            if unchecked is not None:
                unchecked.append(r["file"])
            continue
        path = os.path.join(folder, r["file"].replace("\\", "/"))
        if os.path.exists(path):
            out.append({**r, "label": r["label"].strip(), "path": path})
    return out


def load_rows(folder, unchecked=None):
    import cv2
    rows, unreadable = [], []
    for r in read_labels(folder, unchecked):
        img = cv2.imread(r["path"], cv2.IMREAD_COLOR)
        if img is None or not img.size:
            unreadable.append(r["file"]); continue
        rows.append({**r, "raw": img, "clean": clean_photo(img)})
    return rows, unreadable


def iam_train_texts(words_txt, split_dir, charset_chars, max_len=32):
    """IAM training labels (Aachen train forms, status ok), only to rebuild Step 4's character prior."""
    from src.data.iam import parse_words_txt, word_id_parts
    train = {l.strip() for l in open(os.path.join(split_dir, "train.uttlist")) if l.strip()}
    ok = set(charset_chars)
    return [t for wid, st, t in parse_words_txt(words_txt) if st == "ok" and len(t) <= max_len
            and set(t) <= ok and word_id_parts(wid)[0] in train]


MODEL_A, MODEL_B = "crnn", "crnn_b"     # keys used in the hyps / metrics dicts


def run_crnn(rows, rec, with_raw=True):
    """-> (predictions on the cleaned crops, predictions on the raw crops or None, ms per word)."""
    t = time.time()
    hyp = [rec.predict_tensor(rec.tensor(r["clean"])) if r["clean"] is not None else "" for r in rows]
    raw = [rec.predict_tensor(rec.tensor(to_gray(r["raw"]))) for r in rows] if with_raw else None
    return hyp, raw, 1000 * (time.time() - t) / max(1, (2 if with_raw else 1) * len(rows))


def _norm_cfg(cfg):
    return {k: (list(v) if isinstance(v, (list, tuple)) else v) for k, v in dict(cfg).items()}


def architecture_diff(rec_a, rec_b):
    """Differences between two loaded Recognizers that would make the comparison unfair or the
    weights incompatible: model settings, charset, preprocessing. [] = same architecture."""
    out = []
    a, b = _norm_cfg(rec_a.model_cfg), _norm_cfg(rec_b.model_cfg)
    for k in sorted(set(a) | set(b)):
        if a.get(k) != b.get(k):
            out.append(f"model_cfg.{k}: A={a.get(k)!r} B={b.get(k)!r}")
    if list(rec_a.charset.chars) != list(rec_b.charset.chars):
        out.append("charset differs")
    pa, pb = dict(rec_a.preprocess_cfg), dict(rec_b.preprocess_cfg)
    for k in sorted(set(pa) | set(pb)):
        if pa.get(k) != pb.get(k):
            out.append(f"preprocess.{k}: A={pa.get(k)!r} B={pb.get(k)!r}")
    return out


def config_diff(rec, config):
    """Compare a loaded model with a training run's config.json ({'model': {...}} or the model dict itself):
    every key the config gives must match the model settings stored in the checkpoint."""
    want = _norm_cfg(config.get("model", config))
    have = _norm_cfg(rec.model_cfg)
    return [f"{k}: checkpoint={have.get(k)!r} config.json={v!r}" for k, v in want.items() if have.get(k) != v]


def run_tesseract(rows):
    import pytesseract
    from PIL import Image
    from src.baselines import tesseract as T
    t, out = time.time(), []
    for r in rows:
        if r["clean"] is None:
            out.append(""); continue
        img = T.preprocess(Image.fromarray(r["clean"]))
        out.append(T.normalize(pytesseract.image_to_string(img, lang="eng", config=T.config())))
    return out, 1000 * (time.time() - t) / max(1, len(rows))


def run_emnist(rows, clf, device, params, prior=None, alpha=0.0):
    from src.baselines import char_segment as cs
    from src.baselines.emnist_pipeline import classify
    t, segs, idx = time.time(), [], []
    for i, r in enumerate(rows):
        if r["clean"] is not None:
            pieces, crops = cs.segment_word(r["clean"], params)
            segs.append({"pieces": pieces, "crops": crops}); idx.append(i)
    logps, _ = classify(segs, clf, device) if segs else ([], 0)
    out = [""] * len(rows)
    for i, s, lp in zip(idx, segs, logps):
        out[i] = cs.decode(s["pieces"], lp, prior, alpha)
    return out, 1000 * (time.time() - t) / max(1, len(rows))


def scores(refs, hyps):
    return {"cer": cer(refs, hyps), "wer": wer(refs, hyps),
            "cer_case_insensitive": cer([r.lower() for r in refs], [h.lower() for h in hyps]),
            "word_accuracy": float(np.mean([r == h for r, h in zip(refs, hyps)])) if refs else 0.0,
            "words": len(refs)}

In [ ]:
%%writefile /content/aps360_htr/src/report/webcam_report.py
"""Step 5: report outputs for the webcam test set (words written on paper and captured by a phone
camera with live_demo.py), scored with the same CER/WER code as the IAM results."""
import csv
import json
import random

import numpy as np

from src.report import figures, tables
from src.report.compare import SYSTEMS
from src.report.style import FULL_WIDTH, save_figure

pct = lambda x: f"{100 * x:.2f}"
NAMES = dict((p, n) for n, p in SYSTEMS)            # names used in report_numbers.json["comparison"] (IAM results)
RAW = "crnn_no_cleanup"
MODEL_B = "crnn_b"
ORDER = ["tesseract", "emnist", "crnn", MODEL_B]     # rows of the main webcam table (baselines first)
DISPLAY = {"tesseract": "Tesseract (LSTM)", "emnist": "EMNIST CNN + splitter",
           "crnn": "CRNN, Model A", MODEL_B: "CRNN + augmentation, Model B",
           RAW: "CRNN Model A, raw crop (no clean-up)"}


def save_predictions(rep, rows, hyps):
    """webcam_predictions.csv (one column per model) + webcam_predictions_<model>.csv (one file per model:
    file, label, prediction, exact match, CER of that word)."""
    from src.utils.metrics import cer
    path = rep.dir / "webcam_predictions.csv"
    cols = ["file", "label"] + list(hyps)
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(cols)
        for i, r in enumerate(rows):
            w.writerow([r["file"], r["label"]] + [hyps[k][i] for k in hyps])
    rep.add([str(path)], "Webcam test set: every captured word with its true label and each model's prediction "
                         f"({', '.join(hyps)}; '{RAW}' = CRNN on the raw guide-box crop, without the photo clean-up).")
    per_model = []
    for k, hy in hyps.items():
        pm = rep.dir / f"webcam_predictions_{k}.csv"
        with open(pm, "w", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            w.writerow(["file", "label", "prediction", "correct", "cer"])
            for r, h in zip(rows, hy):
                w.writerow([r["file"], r["label"], h, int(r["label"] == h), f"{cer([r['label']], [h]):.4f}"])
        per_model.append(str(pm))
    rep.add(per_model, "Webcam test set, one file per model: true label, prediction, exact match (1/0) and the "
                       "word's CER.")
    return str(path)


def save_metrics(rep, rows, hyps, ms, info, iam_cer=None):
    """hyps: {'crnn': [...], 'crnn_b': [...], 'crnn_no_cleanup': [...], 'tesseract': [...], 'emnist': [...]}
    (same order as rows; any may be missing). iam_cer: {key: IAM test CER} for models that are not in
    report_numbers.json["comparison"] (Model B, from outputs/model_b/model_b_metrics.json).
    Writes table_webcam_metrics (the 4 models: Tesseract, EMNIST, Model A, Model B) and, if the raw-crop
    row exists, table_webcam_cleanup_ablation (Model A with vs without the photo clean-up)."""
    from src.demo.score import scores
    refs = [r["label"] for r in rows]
    iam = rep.numbers().get("comparison", {})
    iam_cer = dict(iam_cer or {})
    nums = {"words": len(rows), **info, "models": {}}
    main_rows, abl_rows = [], []
    keys = [k for k in ORDER if k in hyps] + ([RAW] if RAW in hyps else [])
    for key in keys:
        s = scores(refs, hyps[key])
        name = DISPLAY[key]
        ic = iam_cer.get(key)
        if ic is None and key in NAMES:
            ic = iam.get(NAMES[key], {}).get("test", {}).get("cer")
        nums["models"][key] = {**s, "name": name, "ms_per_word": ms.get(key), "iam_test_cer": ic,
                               "empty_outputs": sum(1 for h in hyps[key] if not h)}
        row = {"Model": name, "CER": pct(s["cer"]), "WER": pct(s["wer"]),
               "CER no case": pct(s["cer_case_insensitive"]), "Word acc.": pct(s["word_accuracy"]),
               "IAM test CER": pct(ic) if ic is not None else "--"}
        (abl_rows if key == RAW else main_rows).append(row)
        if key == "crnn":
            abl_rows.insert(0, {**row, "Model": "CRNN Model A, cleaned crop (as in the demo)"})
    rep.numbers("webcam", nums)
    (rep.dir / "webcam_metrics.json").write_text(json.dumps(nums, indent=2))
    rep.add([str(rep.dir / "webcam_metrics.json")], "Webcam test set: CER/WER of every model, number of words, settings.")
    cols = ["Model", "CER", "WER", "CER no case", "Word acc.", "IAM test CER"]
    cap = (f"Webcam test set: {len(rows)} handwritten words photographed with a phone camera (DroidCam) and "
           f"captured with the live demo; every model reads the same cleaned-up crop. All numbers in %; CER/WER are "
           f"case-sensitive, lower is better; 'no case' ignores upper/lower case; word acc. = exactly right words. "
           f"IAM test CER (same models, IAM test words) for comparison. Model B = Model A's architecture and "
           f"settings trained with camera-style augmentation.")
    paths = tables.write_table(main_rows, cols, rep.dir, "table_webcam_metrics", cap, "tab:webcam")
    rep.add(paths, "Webcam test set results: Tesseract, EMNIST baseline, CRNN Model A and Model B (augmented) on the "
                   "same captured words, next to their IAM test CER (domain gap).")
    if RAW in hyps and len(abl_rows) == 2:
        p2 = tables.write_table(abl_rows, cols, rep.dir, "table_webcam_cleanup_ablation",
                                "Webcam test set: CRNN Model A on the cleaned crop (as in the live demo) vs on the raw "
                                "guide-box crop (no photo clean-up). All numbers in %.", "tab:webcam_cleanup")
        rep.add(p2, "Photo clean-up ablation on the webcam set (Model A with vs without the clean-up).")
    return nums


def save_vs_iam_figure(rep, nums):
    ms = [(v["name"], v["iam_test_cer"], v["cer"]) for k, v in nums["models"].items() if k != RAW]
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(FULL_WIDTH * 0.7, 2.0))
    x = np.arange(len(ms))
    a = ax.bar(x - 0.18, [100 * (m[1] or 0) for m in ms], 0.36, label="IAM test", color=figures.BLUE)
    b = ax.bar(x + 0.18, [100 * m[2] for m in ms], 0.36, label=f"Webcam ({nums['words']} words)", color=figures.ORANGE)
    for bars in (a, b):
        for r in bars:
            ax.annotate(f"{r.get_height():.1f}", (r.get_x() + r.get_width() / 2, r.get_height()), ha="center",
                        va="bottom", fontsize=6.5, xytext=(0, 1), textcoords="offset points")
    ax.set_xticks(x); ax.set_xticklabels([m[0].replace(" (", "\n(").replace(" + ", "\n+ ") for m in ms], fontsize=7)
    ax.set_ylabel("CER (%)"); ax.legend(frameon=False, fontsize=7); ax.grid(axis="x", visible=False)
    ax.set_ylim(0, max(100 * max((m[1] or 0), m[2]) for m in ms) * 1.18 + 1)
    paths = save_figure(fig, rep.dir, "fig_webcam_vs_iam")
    rep.add(paths, "Character error rate on IAM test words vs on the webcam test set, for each model (domain gap from "
                   "scanned forms to phone-camera photos).")
    return paths


def save_prediction_grid(rep, rows, hyps, n=12, seed=0, name="fig_webcam_predictions", model="CRNN Model A"):
    idx = sorted(random.Random(seed).sample(range(len(rows)), min(n, len(rows))))
    imgs = [rows[i]["clean"] if rows[i]["clean"] is not None else np.full((32, 64), 250, np.uint8) for i in idx]
    refs, hy = [rows[i]["label"] for i in idx], [hyps[i] for i in idx]
    paths = figures.plot_prediction_grid(imgs, refs, hy, rep.dir, name)
    right = sum(r == h for r, h in zip(refs, hy))
    rep.add(paths, f"{model} on {len(idx)} randomly chosen webcam words ({right}/{len(idx)} exactly right), shown "
                   "after the photo clean-up; green = correct, red = wrong.")
    return paths


def save_preprocessing_figure(rep, rows, rec, n=4, seed=1):
    """Raw guide-box crop -> cleaned word -> the 32-px model input, for a few random words."""
    import cv2
    import matplotlib.pyplot as plt
    from src.demo.webcam import tensor_to_u8
    idx = sorted(random.Random(seed).sample(range(len(rows)), min(n, len(rows))))
    fig, axes = plt.subplots(len(idx), 3, figsize=(FULL_WIDTH, 0.75 * len(idx) + 0.3),
                             gridspec_kw={"width_ratios": [1.3, 1, 1]}, squeeze=False)
    for (a, b, c), i in zip(axes, idx):
        r = rows[i]
        a.imshow(cv2.cvtColor(r["raw"], cv2.COLOR_BGR2RGB)); a.set_ylabel(r["label"], rotation=0, ha="right",
                                                                         va="center", fontsize=7.5)
        if r["clean"] is not None:
            b.imshow(r["clean"], cmap="gray", vmin=0, vmax=255)
            c.imshow(tensor_to_u8(rec.tensor(r["clean"])), cmap="gray", vmin=0, vmax=255)
        for ax in (a, b, c):
            ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for ax, t in zip(axes[0], ("camera crop (guide box)", "after clean-up", "model input (32 px high)")):
        ax.set_title(t, fontsize=8)
    fig.tight_layout(h_pad=0.4, w_pad=0.6)
    paths = save_figure(fig, rep.dir, "fig_webcam_preprocessing")
    rep.add(paths, "Webcam preprocessing: the camera crop, after the photo clean-up (lighting flattened, ink found, "
                   "tight crop, IAM-like contrast), and the 32-px input the CRNN reads (same transform as training).")
    return paths

## Install missing packages
Colab already has PyTorch, OpenCV, NumPy, Pillow, matplotlib, pyarrow and pytest, so this only installs what's missing.
`editdistance` (faster CER/WER) is optional; the code falls back to pure Python without it.

In [ ]:
import importlib.util, subprocess, sys
need = {"torch": "torch", "torchvision": "torchvision", "cv2": "opencv-python", "numpy": "numpy",
        "PIL": "pillow", "matplotlib": "matplotlib", "tqdm": "tqdm", "pytest": "pytest",
        "pyarrow": "pyarrow", "huggingface_hub": "huggingface_hub",
        "pytesseract": "pytesseract", "editdistance": "editdistance"}
missing = [pkg for mod, pkg in need.items() if importlib.util.find_spec(mod) is None]
print("missing:", missing or "nothing")
for pkg in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], capture_output=True, text=True)
    note = " (optional: CER/WER use the pure-Python fallback)" if pkg == "editdistance" else ""
    print(f"  {pkg}: {'installed' if r.returncode == 0 else 'install failed' + note}")

## Run the tests
Photo clean-up, export (identical outputs), the demo's keys/captures, the self-contained bundle (run headless), the webcam scoring and report files (Model A + Model B + both baselines, 4-row table, per-model predictions, same-architecture check), and PAGE mode: one commit per word over 30 frames, nothing added while pausing, new words appended, no duplicates under jitter/shifts, growing words updated, the line/word splitter on a synthetic page. Every line should say `passed`.

In [ ]:
!python -m pytest -q tests/test_webcam.py tests/test_webcam_models.py tests/test_camera_switch.py tests/test_mjpeg.py tests/test_page_reader.py tests/test_metrics.py tests/test_report.py

## Find the trained CRNN
This is **Model A** (no augmentation). Uses the newest `runs/crnn_*/best.pt` in your project folder (Step 2 trains `crnn_lr0.001_h256_d0.3_b32_L2`), **ignoring Model B's `runs/crnn_aug_*`** (Step 6). Set `CRNN_RUN` to pick another run.

In [ ]:
import glob, json, os, random, shutil, subprocess, sys, time
import cv2, numpy as np, torch
CRNN_RUN = None          # e.g. "crnn_lr0.001_h256_d0.3_b32_L2"; None = newest run with a best.pt
if CRNN_RUN:
    BEST_PT = os.path.join(PROJECT_DIR, "runs", CRNN_RUN, "best.pt")
else:
    runs = sorted((p for p in glob.glob(os.path.join(PROJECT_DIR, "runs", "crnn_*", "best.pt"))
                   if not os.path.basename(os.path.dirname(p)).startswith("crnn_aug_")), key=os.path.getmtime)
    assert runs, "No runs/crnn_*/best.pt (Model A) found: run Step 2 first."
    BEST_PT = runs[-1]
RUN_NAME = os.path.basename(os.path.dirname(BEST_PT))
CRNN_METRICS = json.load(open(os.path.join(REPORT_DIR, "crnn_metrics.json"))) if os.path.exists(os.path.join(REPORT_DIR, "crnn_metrics.json")) else {}
test = CRNN_METRICS.get("test", {})
assert not RUN_NAME.startswith("crnn_aug_"), "CRNN_RUN must be Model A (Step 2), not Model B"
print("CRNN Model A run:", RUN_NAME, f"({os.path.getsize(BEST_PT) / 1e6:.0f} MB with optimizer state)")
if test:
    print(f"IAM test CER {100 * test['cer']:.2f}% | WER {100 * test['wer']:.2f}% ({test['words']:,} words)")

## Export a CPU-friendly model
`best.pt` also holds the optimizer and scheduler state (~60 MB). The export keeps only what the demo needs: float32 weights (~21 MB), the model settings, the 78-character charset and the preprocessing settings (32 px high, width 16-512). It loads with `torch.load(..., weights_only=True)` and needs no GPU. The check below runs the same random input through the original checkpoint and the export.

In [ ]:
from src.demo import webcam as W, bundle
from src.training.engine import load_model
LOCAL_LIVE = "/content/live_demo"
shutil.rmtree(LOCAL_LIVE, ignore_errors=True); os.makedirs(LOCAL_LIVE)
MODEL_INFO = {"run": RUN_NAME, "iam_test_cer": test.get("cer"), "iam_test_wer": test.get("wer"),
              "code_version": open("/content/aps360_htr/CODE_VERSION").read().strip(),
              "exported": time.strftime("%Y-%m-%d %H:%M")}
exp = W.export_model(BEST_PT, os.path.join(LOCAL_LIVE, "crnn_model.pt"), MODEL_INFO)
MODEL_INFO.update(exp["info"])
ref_model, charset, _ = load_model(BEST_PT, torch.device("cpu"))
rec = W.Recognizer(os.path.join(LOCAL_LIVE, "crnn_model.pt"))
x = torch.rand(2, 1, 32, 160) * 2 - 1
with torch.no_grad():
    diff = (ref_model(x) - rec.model(x)).abs().max().item()
print(f"exported crnn_model.pt: {os.path.getsize(os.path.join(LOCAL_LIVE, 'crnn_model.pt')) / 1e6:.1f} MB | "
      f"epoch {exp['info']['epoch']} | max output difference vs best.pt: {diff:.2e}")
assert diff < 1e-4

## Check: the demo's code gives the evaluation's predictions
`N_CHECK` random IAM **test** words are read straight from `archive.zip` (no unzip) and run through the demo's `Recognizer`, then compared with Step 2's `crnn_test_predictions.csv`:
- **without the photo clean-up** the input tensor is exactly the evaluation's; predictions can still differ on a few words, because evaluation runs in padded batches (the BiLSTM also sees a little white padding) while the demo reads one word at a time. The CER should be about the same.
- **with the photo clean-up** (what the demo does to camera images): IAM scans are already clean, so this mainly shows the clean-up doesn't hurt (on the box, CER rose by ~0.3 points on 400 test words).

In [ ]:
import csv, zipfile
from src.utils.metrics import cer
N_CHECK = 200
CHECK_DIR = "/content/check_words"; os.makedirs(CHECK_DIR, exist_ok=True)
pred_csv = os.path.join(REPORT_DIR, "crnn_test_predictions.csv")
SAMPLES = []
if not os.path.exists(pred_csv):
    print("crnn_test_predictions.csv not found (run Step 2's evaluation); skipping the check.")
else:
    ev = list(csv.DictReader(open(pred_csv, newline="")))
    with zipfile.ZipFile(ARCHIVE) as z:
        members = {os.path.basename(n)[:-4]: n for n in z.namelist() if n.endswith(".png")}
        pick = [r for r in random.Random(0).sample(ev, min(4 * N_CHECK, len(ev))) if r["word_id"] in members][:N_CHECK]
        for r in pick:
            with open(os.path.join(CHECK_DIR, r["word_id"] + ".png"), "wb") as f:
                f.write(z.read(members[r["word_id"]]))
    refs, hyp_eval, hyp_demo, hyp_clean = [], [], [], []
    for r in pick:
        g = cv2.imread(os.path.join(CHECK_DIR, r["word_id"] + ".png"), cv2.IMREAD_GRAYSCALE)
        if g is None:
            continue
        refs.append(r["ref"]); hyp_eval.append(r["hyp"])
        hyp_demo.append(rec.read(g, clean=False)[0]); hyp_clean.append(rec.read(g)[0])
    same = np.mean([a == b for a, b in zip(hyp_demo, hyp_eval)])
    print(f"{len(refs)} IAM test words | demo == evaluation prediction on {100 * same:.1f}% of words")
    print(f"CER: evaluation {100 * cer(refs, hyp_eval):.2f}% | demo {100 * cer(refs, hyp_demo):.2f}% | "
          f"demo with photo clean-up {100 * cer(refs, hyp_clean):.2f}%")
    good = [r for r in pick if r["ref"] == r["hyp"] and len(r["ref"]) >= 4][:3]
    SAMPLES = [(os.path.join(CHECK_DIR, r["word_id"] + ".png"), r["ref"]) for r in good]

## Build `live_demo` on Drive (+ `live_demo.zip`)
The folder is built on Colab's disk, tested by running `live_demo.py` **from inside the folder** (headless, on a sample word, so it only uses the bundled code), then copied to `live_demo/` in your project folder and zipped as `live_demo.zip` next to it. Existing files in `live_demo/` on Drive that aren't ours (e.g. a `webcam_test` you put there) are never deleted.

In [ ]:
files = bundle.build(LOCAL_LIVE, os.path.join(LOCAL_LIVE, "crnn_model.pt"), MODEL_INFO, SAMPLES)
if SAMPLES:
    env = {k: v for k, v in os.environ.items() if k != "PYTHONPATH"}
    r = subprocess.run([sys.executable, "live_demo.py", "--image", os.path.join("samples", os.path.basename(SAMPLES[0][0])),
                        "--headless", "--max-frames", "3", "--box", "0.98", "0.98"], cwd=LOCAL_LIVE, env=env,
                       capture_output=True, text=True)
    print(r.stdout[-600:] or r.stderr[-1500:])
    print("true word:", SAMPLES[0][1])
LIVE_DIR = os.path.join(PROJECT_DIR, "live_demo")
for f in files:
    os.makedirs(os.path.dirname(os.path.join(LIVE_DIR, f)), exist_ok=True)
    shutil.copy2(os.path.join(LOCAL_LIVE, f), os.path.join(LIVE_DIR, f))
LIVE_ZIP = os.path.join(PROJECT_DIR, "live_demo.zip")
shutil.copy(bundle.make_zip(LOCAL_LIVE, "/content/live_demo.zip"), LIVE_ZIP)
print(f"\n{len(files)} files in {LIVE_DIR}")
for f in files:
    print(f"  {os.path.getsize(os.path.join(LOCAL_LIVE, f)) / 1024:9.1f} KB  {f}")
print(f"live_demo.zip: {os.path.getsize(LIVE_ZIP) / 1e6:.1f} MB  ->  {LIVE_ZIP}")

## On your laptop (short version; full steps in the folder's README.md)
1. Install Python 3.12 from python.org (tick **Add python.exe to PATH**).
2. Download `live_demo.zip` from the project folder on Drive, right-click → *Extract All*.
3. In the extracted folder: address bar → `cmd` → `python -m pip install -r requirements.txt`.
4. DroidCam app on the phone + DroidCam Client on Windows, same Wi-Fi, type the phone's IP, **Start**.
5. `python live_demo.py`. WORD mode: `c` = capture + type the true word + Enter; `space` freeze; `a/d/w/s` box size; `n` next camera; `m` PAGE mode (beta); `q` quit.
6. Collect ~150 words, then drag the `webcam_test` folder into the project folder on Drive and run this notebook again.

## Load Model B (augmented CRNN from Step 6)
Model A is the `crnn_model.pt` exported above (the same file as `live_demo/crnn_model.pt`). Model B is **`outputs/model_b/crnn_model_b.pt`**, saved by Step 6. Before scoring, the notebook checks that Model B has **the same architecture** as Model A (CNN channels, LSTM size/layers, dropout), the same 78-character charset and the same preprocessing, and that it matches its own training run's `config.json` (`runs/crnn_aug_*/`). Only the training data differ (camera-style augmentation), so A vs B isolates the augmentation. If Step 6 hasn't run, Model B is skipped and the table has 3 rows.

In [ ]:
from src.demo import score as S
MODEL_B_PT = os.path.join(OUTPUT_DIR, "model_b", "crnn_model_b.pt")
MODEL_B_METRICS = os.path.join(OUTPUT_DIR, "model_b", "model_b_metrics.json")
rec_b, MODEL_B_INFO = None, {}
IAM_CER = {"crnn": CRNN_METRICS["test"]["cer"]} if "cer" in CRNN_METRICS.get("test", {}) else {}
if not os.path.exists(MODEL_B_PT):
    print("Model B not found:", MODEL_B_PT, "(run Step 6 first); scoring Model A and the baselines only.")
else:
    rec_b = W.Recognizer(MODEL_B_PT)
    diffs = S.architecture_diff(rec, rec_b)
    assert not diffs, "Model B's architecture differs from Model A's: " + "; ".join(diffs)
    run_b = rec_b.info.get("run") or ""
    cfg_paths = [os.path.join(PROJECT_DIR, "runs", run_b, "config.json")] if run_b else []
    cfg_paths += sorted(glob.glob(os.path.join(PROJECT_DIR, "runs", "crnn_aug_*", "config.json")), key=os.path.getmtime)[::-1]
    cfg_path = next((p for p in cfg_paths if os.path.exists(p)), None)
    if cfg_path:
        cdiff = S.config_diff(rec_b, json.load(open(cfg_path)))
        assert not cdiff, f"crnn_model_b.pt doesn't match {cfg_path}: " + "; ".join(cdiff)
        print("Model B matches its run's config.json:", os.path.relpath(cfg_path, PROJECT_DIR))
    else:
        print("note: no runs/crnn_aug_*/config.json found; checked against Model A only")
    if os.path.exists(MODEL_B_METRICS):
        mb_test = json.load(open(MODEL_B_METRICS)).get("test", {})
        if "cer" in mb_test:
            IAM_CER["crnn_b"] = mb_test["cer"]
            print(f"Model B IAM test CER {100 * mb_test['cer']:.2f}% | WER {100 * mb_test['wer']:.2f}%")
    MODEL_B_INFO = {"file": os.path.relpath(MODEL_B_PT, PROJECT_DIR), **rec_b.info}
    print(f"Model B loaded: run {run_b or '?'}, epoch {rec_b.info.get('epoch')} | same architecture as Model A:",
          rec_b.model_cfg)

# Score the webcam test set
Looks for a folder named `webcam_test` (with `labels.csv`) in your project folder (also one level down, e.g. inside `live_demo/`). Every model reads **the same cleaned crop** of each word (the same `clean_photo` as the live demo):
- **CRNN Model A** (Step 2) and **CRNN Model B** (Step 6, augmented): training transform + greedy CTC, exactly as in the demo; an extra **ablation** table reads the raw guide-box crop with Model A, without the clean-up;
- **Tesseract**: Step 3's settings (16-px white border, `--oem 1 --psm 8`);
- **EMNIST baseline**: Step 4's splitter settings and IAM character prior (chosen on IAM validation), if Step 4 has run.

Labels are compared **case-sensitively** with the same CER/WER code as before. Rows with `needs_check = 1` (PAGE-mode page captures labelled with the model's guess) are **not scored** until you fix the label and set it to 0.

In [ ]:
dirs = S.find_webcam_dirs(PROJECT_DIR)
WEBCAM_DIR, WEBCAM_ROWS, UNCHECKED = (dirs[0] if dirs else None), [], []
if WEBCAM_DIR is None:
    print("No webcam_test/ folder with labels.csv in", PROJECT_DIR)
    print("""Nothing to score yet. To make the webcam test set:
  1. Download live_demo.zip (project folder on Drive), extract it, and run:  python live_demo.py
  2. For each word: hold it in the green box, press c, type the true word, press Enter (~150 words).
  3. The captures are in the folder webcam_test next to live_demo.py (images/ + labels.csv).
  4. Drag the whole webcam_test folder into the project folder on Drive (next to archive.zip).
  5. Run this notebook again (Runtime -> Run all).""")
else:
    WEBCAM_ROWS, unreadable = S.load_rows(WEBCAM_DIR, UNCHECKED)
    print(f"{WEBCAM_DIR}: {len(WEBCAM_ROWS)} labelled words" + (f" (also found: {dirs[1:]})" if len(dirs) > 1 else ""))
    if UNCHECKED:
        print(f"  {len(UNCHECKED)} page-capture rows still have needs_check=1 (label is only a guess): not scored")
    if unreadable:
        print(f"  {len(unreadable)} unreadable image(s) skipped: {unreadable[:10]}")
    no_ink = [r["file"] for r in WEBCAM_ROWS if r["clean"] is None]
    if no_ink:
        print(f"  {len(no_ink)} capture(s) with no clear writing in the box (they score as empty predictions): {no_ink[:10]}")
    outside = sorted({c for r in WEBCAM_ROWS for c in r["label"] if c not in rec.charset.chars})
    if outside:
        print("  characters in the labels that the CRNN can't output:", outside)
    if len(WEBCAM_ROWS) < 100:
        print(f"  note: only {len(WEBCAM_ROWS)} words; the proposal plans ~150, so these numbers are noisy.")

### Run the four models on the same words

In [ ]:
from src.report import webcam_report as wr
HYPS, MS, BASE_INFO = {}, {}, {}
if WEBCAM_ROWS:
    HYPS["crnn"], HYPS[wr.RAW], MS["crnn"] = S.run_crnn(WEBCAM_ROWS, rec)
    print(f"CRNN Model A done ({MS['crnn']:.1f} ms/word on the CPU)")
    if rec_b is not None:
        HYPS["crnn_b"], _, MS["crnn_b"] = S.run_crnn(WEBCAM_ROWS, rec_b, with_raw=False)
        print(f"CRNN Model B done ({MS['crnn_b']:.1f} ms/word on the CPU)")
    !apt-get -qq install -y tesseract-ocr > /dev/null
    !pip -q install pytesseract
    try:
        from src.baselines import tesseract as T
        HYPS["tesseract"], MS["tesseract"] = S.run_tesseract(WEBCAM_ROWS)
        BASE_INFO["tesseract"] = {"version": T.version(), "config": T.config(), "border_px": T.BORDER}
        print(f"Tesseract done ({MS['tesseract']:.0f} ms/word)")
    except Exception as e:
        print("Tesseract skipped:", e)
    from src.baselines import emnist as E, char_segment as cs
    em_ck = os.path.join(PROJECT_DIR, "runs", "emnist_cnn", "best.pt")
    em_json = os.path.join(REPORT_DIR, "emnist_metrics.json")
    em_labels = os.path.join(PROJECT_DIR, "emnist", E.FILES["train_y"])
    if os.path.exists(em_ck) and os.path.exists(em_json):
        st = json.load(open(em_json))["settings"]
        params = cs.SegParams(**{k: st[k] for k in cs.params_dict(cs.SegParams()) if k in st})
        prior, alpha = None, 0.0
        words_txt = OFFICIAL_WORDS_TXT if os.path.exists(OFFICIAL_WORDS_TXT) else RECOVERED_WORDS_TXT
        if st.get("iam_prior") and os.path.exists(em_labels) and os.path.exists(words_txt):
            texts = S.iam_train_texts(words_txt, "/content/aps360_htr/data/splits/aachen", rec.charset.chars)
            prior, alpha = cs.char_prior(texts, E.read_idx(em_labels)), 1.0
        clf = E.load_classifier(em_ck, torch.device("cpu"))
        HYPS["emnist"], MS["emnist"] = S.run_emnist(WEBCAM_ROWS, clf, torch.device("cpu"), params, prior, alpha)
        BASE_INFO["emnist"] = {"settings": st, "iam_prior_used": bool(alpha)}
        print(f"EMNIST baseline done ({MS['emnist']:.1f} ms/word; width factor {params.width_factor}, "
              f"IAM prior {'on' if alpha else 'off'})")
    else:
        print("EMNIST baseline skipped: run Step 4 first (needs runs/emnist_cnn/best.pt and emnist_metrics.json).")

# Report outputs
Saved to `outputs/report/` (same folder as Steps 2-4): figures as PNG (300 dpi) + PDF, tables as CSV + LaTeX booktabs `.tex`.

In [ ]:
from IPython.display import Image, Markdown, display
from src.report.manifest import Report, future_work
from src.report.tables import csv_to_markdown
rep = Report(REPORT_DIR, step=5)
def show(paths):
    for p in paths or []:
        if p.endswith(".png"):
            display(Image(p, width=700))
show_table = lambda name: display(Markdown(csv_to_markdown(os.path.join(REPORT_DIR, name + ".csv"))))
if WEBCAM_ROWS:
    wr.save_predictions(rep, WEBCAM_ROWS, HYPS)
    info = {"folder": os.path.relpath(WEBCAM_DIR, PROJECT_DIR), "crnn_run": RUN_NAME, "model_b": MODEL_B_INFO,
            "unchecked_skipped": len(UNCHECKED),
            "no_ink": sum(r["clean"] is None for r in WEBCAM_ROWS), "baselines": BASE_INFO,
            "device": "CPU", "case_sensitive": True}
    WEBCAM_NUMS = wr.save_metrics(rep, WEBCAM_ROWS, HYPS, MS, info, iam_cer=IAM_CER)
    show_table("table_webcam_metrics")
    print("Photo clean-up ablation (Model A):")
    show_table("table_webcam_cleanup_ablation")
    print("Per-word predictions:", ", ".join(f"webcam_predictions_{k}.csv" for k in HYPS))
else:
    print("No webcam words yet: report outputs skipped (see the instructions above).")

### Figures
`fig_webcam_vs_iam` (CER on IAM test vs webcam, per model), `fig_webcam_predictions` (12 random webcam words, Model A), `fig_webcam_predictions_model_b` (same words, Model B), `fig_webcam_preprocessing` (camera crop → clean-up → model input).

In [ ]:
if WEBCAM_ROWS:
    show(wr.save_vs_iam_figure(rep, WEBCAM_NUMS))
    show(wr.save_prediction_grid(rep, WEBCAM_ROWS, HYPS["crnn"]))
    if "crnn_b" in HYPS:
        show(wr.save_prediction_grid(rep, WEBCAM_ROWS, HYPS["crnn_b"], name="fig_webcam_predictions_model_b",
                                     model="CRNN Model B"))
    show(wr.save_preprocessing_figure(rep, WEBCAM_ROWS, rec))

### Summary and list of files

In [ ]:
summary = rep.write_summary(future_work(rep))
display(Markdown(open(summary).read()))
print("Saved report files:")
rep.list_files();

## Notes for the report
- **Domain gap:** IAM is scanned forms; the webcam set is phone-camera photos (blur, uneven light, JPEG, different pens and writers). Expect a higher CER than on IAM for every model; `table_webcam_metrics` puts both side by side.
- **Same input for every model:** all four read the same cleaned crop, so the comparison is fair. `table_webcam_cleanup_ablation` (Model A, raw crop) shows how much the clean-up itself is worth.
- **Model A vs Model B:** same architecture, charset and preprocessing (checked above); only the training augmentation differs, so the webcam CER gap between the two rows is the effect of augmentation. On clean IAM test, B is already better (8.77% vs 10.75% CER).
- **Small test set:** ~150 words from few writers, so ±several points of CER is noise. Say how many words and writers.
- **Preprocessing = training:** the demo and this notebook use the training transform unchanged (32 px high, aspect kept, [-1, 1]); only the photo clean-up before it is new.
- **PAGE mode is a beta extra:** it is not used for any number in the report.
- **Per-word predictions:** `webcam_predictions.csv` (all models side by side) and `webcam_predictions_<model>.csv` (one per model: file, label, prediction, correct, cer).